# Campus Courier — pipeline end-to-end

Pipeline inference hoàn chỉnh với hai checkpoint đã huấn luyện: NLU 7-kind và YOLO 42-class.

1. CELL 1 khai báo đường dẫn dữ liệu và hai model.
2. Nạp JSON, khởi tạo NLU, nạp YOLO→graph và bộ mô phỏng chiến thuật công khai.
3. Validation chấm riêng NLU, graph, route demo, strategy sweep trên graph chuẩn, và end-to-end từ ảnh/yêu cầu.
4. Test chạy YOLO + NLU trên 5 demo và 6 query mỗi episode, lọc họ chiến thuật từ demo, rồi ghi `predictions.json` theo thứ tự queries.

Notebook không fine-tune lại YOLO/NLU; nó dùng checkpoint bạn đã có. `strategies.json` và labels validation chỉ phục vụ audit/chấm, không tham gia fit robot cho test. Nếu checkpoint YOLO đã chuyển sang Drive, sửa `YOLO_MODEL_PATH` ở CELL 1.

In [ ]:
# CELL 1 — Cấu hình duy nhất: dữ liệu và 2 checkpoint đang dùng
%pip -q install "ultralytics>=8.3" "transformers>=4.44,<5" sentencepiece accelerate opencv-python-headless matplotlib pyyaml tqdm

from pathlib import Path
import os, json, zipfile, shutil, math, random, statistics, hashlib, re, copy, unicodedata, tempfile

# Google Colab: mount Drive khi chưa có file dữ liệu.
DATA_ZIP_PATH = Path("/content/drive/MyDrive/FINAL1_data.zip")
try:
    from google.colab import drive
    if not DATA_ZIP_PATH.is_file():
        drive.mount("/content/drive")
except ImportError:
    pass
if not DATA_ZIP_PATH.is_file() and Path("./FINAL1_data.zip").is_file():
    DATA_ZIP_PATH = Path("./FINAL1_data.zip")
if not DATA_ZIP_PATH.is_file():
    raise FileNotFoundError(f"Không tìm thấy FINAL1_data.zip: {DATA_ZIP_PATH}")

# NLP: checkpoint Phase 12 trên Drive; CELL 5 remap 12-kind -> schema 7-kind.
NLU_MODEL_PATH = Path(
    "/content/drive/MyDrive/bamibert_nlu_phase12_v3_4views/"
    "bamibert_nlu_phase12_v3_4views_best.pt"
)

# YOLO: checkpoint 42-class hiện có trong runtime Colab.
YOLO_MODEL_PATH = Path("/content/best (5).pt")

MODEL_NAME = "Qualcomm-AI-Research/BamiBERT"
MAX_LEN = 256
EXTRA_POOLS = 3
EVAL_BATCH_SIZE = 32

OUTPUT_DIR = Path("./courier_complete_pipeline_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_CACHE_DIR = OUTPUT_DIR / "image_cache"
IMAGE_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Ngưỡng dùng cho parser 42 lớp; route có thể có confidence thấp hơn cạnh map.
# Route detections are drawn in a lane offset beside each main road edge.
ROUTE_LANE_OFFSET_RATIO = 0.10
ROUTE_CENTER_T = 0.50
ROUTE_BOX_LENGTH_RATIO = 0.76
ROUTE_BOX_WIDTH_RATIO = 0.16
ROUTE_COLOR_MAX_DIST = 42.0
ROUTE_COLOR_MIN_PIXELS = 5
ROUTE_USE_LEGEND_COLOR = True

YOLO_CONF = 0.20
YOLO_EDGE_CONF = 0.20
YOLO_ROUTE_CONF = 0.12
YOLO_IMGSZ = 1024
SEED = 15102026
random.seed(SEED)

for label, path in (("NLU", NLU_MODEL_PATH), ("YOLO", YOLO_MODEL_PATH)):
    if not path.is_file():
        raise FileNotFoundError(
            f"Không tìm thấy checkpoint {label}: {path}\n"
            "Sửa đường dẫn tương ứng ở CELL 1."
        )
print("Data:", DATA_ZIP_PATH)
print("NLU checkpoint:", NLU_MODEL_PATH)
print("YOLO checkpoint:", YOLO_MODEL_PATH)
print("Output:", OUTPUT_DIR.resolve())

In [ ]:
# CELL 2 — Imports, seed, exact output schema
import os, re, json, math, copy, random, zipfile, hashlib, unicodedata
from collections import Counter, defaultdict
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from tqdm.auto import tqdm

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = DEVICE.type == 'cuda'
AMP_DTYPE = torch.bfloat16 if (USE_AMP and torch.cuda.is_bf16_supported()) else torch.float16

PLACES = ['canteen','clinic','dorm','gate','lab','lecture','library','office','parking','sports']
KINDS = ['none','near','far','north','south','east','west']
HEADS = {
    'goal': PLACES + ['none'],
    'via': ['none'] + PLACES,
    'gkind': KINDS,
    'ganchor': ['none'] + PLACES,
    'vkind': KINDS,
    'vanchor': ['none'] + PLACES,
    'urgent': ['no','yes'],
    'fragile': ['no','yes'],
}
H2I = {h:{v:i for i,v in enumerate(labels)} for h,labels in HEADS.items()}
assert len(KINDS) == 7
print('device:', DEVICE, '| schema:', {h:len(v) for h,v in HEADS.items()})


In [ ]:
# CELL 3 — Nạp JSON train/validation/test trực tiếp từ ZIP

def _zip_json(split, filename):
    suffix = f"{split}/{filename}"
    with zipfile.ZipFile(DATA_ZIP_PATH, "r") as zf:
        names = [name.replace("\\", "/") for name in zf.namelist()]
        matches = [name for name in names if name.endswith(suffix)]
        preferred = [name for name in matches if f"delivery_public/{suffix}" in name]
        matches = preferred or matches
        if not matches:
            raise FileNotFoundError(f"ZIP thiếu {suffix}: {DATA_ZIP_PATH}")
        shortest = min(map(len, matches))
        best = [name for name in matches if len(name) == shortest]
        if len(best) != 1:
            raise ValueError(f"Có nhiều member ZIP phù hợp {suffix}: {best}")
        return json.loads(zf.read(best[0]).decode("utf-8"))

def _as_list(value, key=None):
    if isinstance(value, list):
        return value
    if isinstance(value, dict) and key and isinstance(value.get(key), list):
        return value[key]
    raise ValueError(f"JSON không có list{f' ở khóa {key!r}' if key else ''}")

TRAIN_SCENES = _as_list(_zip_json("train", "scenes.json"), "scenes")
VAL_SCENES = _as_list(_zip_json("validation", "scenes.json"), "scenes")
TRAIN_EPISODES = _as_list(_zip_json("train", "episodes.json"), "episodes")
VAL_EPISODES = _as_list(_zip_json("validation", "episodes.json"), "episodes")
VAL_QUERIES = _as_list(_zip_json("validation", "queries.json"), "queries")
VAL_LABELS = _as_list(_zip_json("validation", "labels.json"), "labels")
VAL_STRATEGIES = _zip_json("validation", "strategies.json")
TEST_EPISODES = _as_list(_zip_json("test", "episodes.json"), "episodes")
TEST_QUERIES = _as_list(_zip_json("test", "queries.json"), "queries")

assert len(VAL_QUERIES) == len(VAL_LABELS), (len(VAL_QUERIES), len(VAL_LABELS))
assert all(len(ep.get("demos", [])) == 5 for ep in TEST_EPISODES)
assert len(TEST_QUERIES) > 0
print(f"train scenes={len(TRAIN_SCENES)} | validation scenes={len(VAL_SCENES)}")
print(f"validation episodes={len(VAL_EPISODES)} queries={len(VAL_QUERIES)}")
print(f"test episodes={len(TEST_EPISODES)} queries={len(TEST_QUERIES)}")

In [ ]:
# CELL 4 — Chuyển mission có nhãn thành dữ liệu đánh giá NLU
def make_row(scene, split):
    m = scene.get('mission') or {}
    text = str(m.get('text') or '').strip()
    if not text: raise ValueError(f"Mission text rỗng: {scene.get('scene_id')}")
    g = m.get('goal_ref') or {}; v = m.get('via_ref') or {}
    gkind = g.get('kind') or 'none'; vkind = v.get('kind') or 'none'
    ganchor = g.get('anchor') or 'none'; vanchor = v.get('anchor') or 'none'
    goal = m.get('goal'); via = m.get('via') or 'none'
    # Task mới giao tới một địa điểm có tên; tham chiếu chỉ định bản/vị trí của địa điểm đó.
    if goal not in PLACES: raise ValueError((scene.get('scene_id'),'goal',goal,m))
    if via not in PLACES + ['none']: raise ValueError((scene.get('scene_id'),'via',via,m))
    if gkind not in KINDS or vkind not in KINDS: raise ValueError((scene.get('scene_id'),'kind',gkind,vkind,m))
    if gkind in ('near','far'):
        if ganchor not in PLACES: raise ValueError((scene.get('scene_id'),'goal anchor',ganchor,m))
    elif ganchor != 'none': raise ValueError((scene.get('scene_id'),'unexpected goal anchor',gkind,ganchor,m))
    if via == 'none':
        if vkind != 'none' or vanchor != 'none': raise ValueError((scene.get('scene_id'),'waypoint without via',m))
    elif vkind in ('near','far'):
        if vanchor not in PLACES: raise ValueError((scene.get('scene_id'),'via anchor',vanchor,m))
    elif vanchor != 'none': raise ValueError((scene.get('scene_id'),'unexpected via anchor',vkind,vanchor,m))
    values = {'goal':goal,'via':via,'gkind':gkind,'ganchor':ganchor,
              'vkind':vkind,'vanchor':vanchor,
              'urgent':'yes' if bool(m.get('urgent',False)) else 'no',
              'fragile':'yes' if bool(m.get('fragile',False)) else 'no'}
    return {'text':text,'y':{h:H2I[h][values[h]] for h in HEADS},
            'values':values,'scene_id':scene.get('scene_id'),'split':split}

TRAIN_BASE = [make_row(scene, "train") for scene in TRAIN_SCENES]
VAL_BASE = [make_row(scene, "validation") for scene in VAL_SCENES]
assert len({row["scene_id"] for row in TRAIN_BASE}) == len(TRAIN_BASE)
assert len({row["scene_id"] for row in VAL_BASE}) == len(VAL_BASE)
print("NLU labeled missions:", len(TRAIN_BASE), "train |", len(VAL_BASE), "validation")
for field in HEADS:
    print(field, Counter(row["values"][field] for row in TRAIN_BASE))

In [ ]:
# CELL 4 — Safe text views (meaning-preserving)
_BRACKET = re.compile(r'\[[^\]]*\]')
def strip_accents(text):
    text = str(text).replace('đ','d').replace('Đ','D')
    return ''.join(c for c in unicodedata.normalize('NFD',text) if unicodedata.category(c) != 'Mn')
def clean_order_ids(text):
    return re.sub(r'\s+',' ',_BRACKET.sub(' ',text)).strip() or text
def four_views(text):
    base = clean_order_ids(text)
    return [text, base, strip_accents(text), strip_accents(base)]

def expand_views(rows):
    return [{'text':view,'y':row['y'],'base_id':row['scene_id'],'view':vi}
            for row in rows for vi,view in enumerate(four_views(row['text']))]

TRAIN_ROWS = expand_views(TRAIN_BASE)
VAL_ROWS = expand_views(VAL_BASE)
# Không thêm route/ảnh hoặc test vào NLU; mỗi view giữ nguyên nhãn semantic gốc.
print('train text views:',len(TRAIN_ROWS),'| validation text views:',len(VAL_ROWS))
print('sample views:')
for t in four_views(TRAIN_BASE[0]['text']): print(' •',t[:180])


In [ ]:
# CELL 5 — Model architecture + checkpoint transfer by label name
class MultiHeadNLU(nn.Module):
    def __init__(self, model_name=MODEL_NAME, extra_pools=EXTRA_POOLS):
        super().__init__()
        self.enc = AutoModel.from_pretrained(model_name)
        hidden = self.enc.config.hidden_size
        self.extra_pools = extra_pools
        self.drop = nn.Dropout(0.2)
        self.pool_q = nn.ModuleDict({h:nn.Sequential(nn.Linear(hidden,hidden//2),nn.Tanh(),nn.Linear(hidden//2,1)) for h in HEADS})
        self.heads = nn.ModuleDict({h:nn.Linear(hidden*2,len(labels)) for h,labels in HEADS.items()})
        self.extra_pool_q = nn.ModuleDict({h:nn.Linear(hidden,extra_pools,bias=False) for h in HEADS})
        extra_size = hidden*(extra_pools+1)
        self.extra_norm = nn.ModuleDict({h:nn.LayerNorm(extra_size) for h in HEADS})
        self.extra_heads = nn.ModuleDict({h:nn.Linear(extra_size,len(labels)) for h,labels in HEADS.items()})
        for h in HEADS:
            nn.init.zeros_(self.pool_q[h][2].weight); nn.init.zeros_(self.pool_q[h][2].bias)
            nn.init.normal_(self.extra_pool_q[h].weight,mean=0.0,std=0.02)
            nn.init.zeros_(self.extra_heads[h].weight); nn.init.zeros_(self.extra_heads[h].bias)

    def forward(self, ids, mask):
        hidden = self.enc(input_ids=ids,attention_mask=mask).last_hidden_state
        cls = hidden[:,0,:]; out = {}; token_mask = mask.unsqueeze(-1)==0
        for h in HEADS:
            score = self.pool_q[h](hidden).squeeze(-1).float().masked_fill(mask==0,-1e4)
            weight = score.softmax(-1).to(hidden.dtype)
            pooled = (weight.unsqueeze(-1)*hidden).sum(1)
            old_logits = self.heads[h](self.drop(torch.cat([cls,pooled],-1)))
            ex_score = self.extra_pool_q[h](hidden).float().masked_fill(token_mask,-1e4)
            ex_weight = ex_score.softmax(1).to(hidden.dtype)
            ex_pool = torch.einsum('btk,bth->bkh',ex_weight,hidden).flatten(1)
            feature = self.extra_norm[h](torch.cat([cls,ex_pool],-1))
            out[h] = old_logits + self.extra_heads[h](self.drop(feature))
        return out

def load_and_remap_checkpoint(path, model):
    ckpt = torch.load(path,map_location='cpu',weights_only=False)
    if not isinstance(ckpt,dict) or 'state' not in ckpt or 'heads' not in ckpt:
        raise ValueError('Checkpoint cần dict có keys state và heads.')
    old_heads = ckpt['heads']; old_state = ckpt['state']
    if set(old_heads) != set(HEADS): raise ValueError(f'Checkpoint head mismatch: {list(old_heads)}')
    for h in HEADS:
        missing = set(HEADS[h])-set(old_heads[h])
        if missing: raise ValueError(f'{h}: checkpoint thiếu nhãn dùng chung {missing}')
    new_state = model.state_dict()
    remapped = []
    for key,target in new_state.items():
        source = old_state.get(key)
        if source is None:
            raise KeyError(f'Checkpoint thiếu tensor {key}')
        if key in ('heads.gkind.weight','heads.gkind.bias','extra_heads.gkind.weight','extra_heads.gkind.bias',
                   'heads.vkind.weight','heads.vkind.bias','extra_heads.vkind.weight','extra_heads.vkind.bias'):
            h = key.split('.')[1]
            for new_i,label in enumerate(HEADS[h]):
                old_i = old_heads[h].index(label)
                target[new_i].copy_(source[old_i])
            remapped.append((h,list(old_heads[h]),list(HEADS[h])))
        elif tuple(source.shape) == tuple(target.shape):
            target.copy_(source)
        else:
            raise ValueError(f'Shape mismatch {key}: checkpoint {tuple(source.shape)} vs model {tuple(target.shape)}')
    model.load_state_dict(new_state,strict=True)
    print('Loaded source phase:',ckpt.get('phase'),'| epoch:',ckpt.get('epoch'))
    print('Kind head remap:',remapped)
    print('Dropped labels:',{h:[x for x in old_heads[h] if x not in HEADS[h]] for h in ('gkind','vkind')})
    return ckpt

# Load and remap the selected checkpoint after defining the seven-kind architecture.
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
model = MultiHeadNLU().to(DEVICE)
SOURCE_CKPT = load_and_remap_checkpoint(NLU_MODEL_PATH, model)
model.eval()
print("NLU checkpoint loaded; 7-kind schema ready.")

In [ ]:
# CELL 6 — Constrained decoder + metrics
def log_probs(logits):
    return {h:F.log_softmax(v.float(),dim=-1).cpu().numpy() for h,v in logits.items()}

def best_ref(kind_lp,anchor_lp):
    best=(-1e30,'none','none')
    for kind in KINDS:
        anchors = PLACES if kind in ('near','far') else ['none']
        for anchor in anchors:
            score=float(kind_lp[H2I['gkind'][kind]]+anchor_lp[H2I['ganchor'][anchor]])
            if score>best[0]: best=(score,kind,anchor)
    return best

def constrained_decode(lp):
    n=len(lp['goal']); preds=[]
    for i in range(n):
        # Goal is always a named campus place in this task; its spatial reference selects a copy/location.
        gbest=None
        for goal in PLACES:
            ref_score,gkind,ganchor=best_ref(lp['gkind'][i],lp['ganchor'][i])
            score=float(lp['goal'][i,H2I['goal'][goal]]+ref_score)
            cand=(score,goal,gkind,ganchor)
            if gbest is None or score>gbest[0]: gbest=cand
        # Via may be absent; when absent its reference heads must also be none.
        vnone=(float(lp['via'][i,H2I['via']['none']]+lp['vkind'][i,H2I['vkind']['none']]+lp['vanchor'][i,H2I['vanchor']['none']]),'none','none','none')
        vbest=vnone
        for via in PLACES:
            kind_lp=lp['vkind'][i]; anchor_lp=lp['vanchor'][i]
            ref_best=(-1e30,'none','none')
            for kind in KINDS:
                anchors=PLACES if kind in ('near','far') else ['none']
                for anchor in anchors:
                    score=float(kind_lp[H2I['vkind'][kind]]+anchor_lp[H2I['vanchor'][anchor]])
                    if score>ref_best[0]: ref_best=(score,kind,anchor)
            score=float(lp['via'][i,H2I['via'][via]]+ref_best[0])
            if score>vbest[0]: vbest=(score,via,ref_best[1],ref_best[2])
        attrs={h:HEADS[h][int(lp[h][i].argmax())] for h in ('urgent','fragile')}
        preds.append({'goal':gbest[1],'gkind':gbest[2],'ganchor':gbest[3],
                      'via':vbest[1],'vkind':vbest[2],'vanchor':vbest[3],**attrs})
    return preds

@torch.inference_mode()
def predict_rows(rows,batch_size=EVAL_BATCH_SIZE):
    model.eval(); outs=[]
    for start in range(0,len(rows),batch_size):
        chunk=rows[start:start+batch_size]
        enc=tok([r['text'] for r in chunk],padding=True,truncation=True,max_length=MAX_LEN,return_tensors='pt').to(DEVICE)
        with torch.autocast(DEVICE.type,dtype=AMP_DTYPE,enabled=USE_AMP):
            logits=model(enc['input_ids'],enc['attention_mask'])
        lp=log_probs(logits); outs.extend(constrained_decode(lp))
    return outs

def evaluate(rows,base_rows):
    pred=predict_rows(rows); exact=Counter(); total=Counter(); per_head=Counter(); n=len(rows)
    for r,p in zip(rows,pred):
        truth={h:HEADS[h][r['y'][h]] for h in HEADS}
        view=r['view']; total[view]+=1
        exact[view]+=int(all(p[h]==truth[h] for h in HEADS))
        for h in HEADS: per_head[(view,h)]+=int(p[h]==truth[h])
    result={v:exact[v]/max(1,total[v]) for v in total}
    print('Exact 8-head by view:',{v:round(result[v],4) for v in sorted(result)})
    for view in sorted(total):
        print(view,'head acc:',{h:round(per_head[(view,h)]/total[view],4) for h in HEADS})
    return float(np.mean(list(result.values()))),result,pred

BASELINE_SCORE,BASELINE_BY_VIEW,VAL_PRED=evaluate(VAL_ROWS,VAL_BASE)
print('Baseline mean exact8:',round(BASELINE_SCORE,5))


In [ ]:
import json
import math
import os
from collections import Counter, defaultdict

import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from ultralytics import YOLO

# ==========================================
# CONFIG
# ==========================================
CLASSES = [
    'canteen', 'clinic', 'dorm', 'gate', 'lab', 'lecture', 'library', 'office', 'parking', 'sports',
    'robot_up', 'robot_down', 'robot_left', 'robot_right',
    'edge_type_1', 'edge_type_2', 'edge_type_3',
    'print_edge_1', 'print_edge_2', 'print_edge_3',
    'closed', 'stairs', 'oneway_up', 'oneway_down', 'oneway_left', 'oneway_right',
    'weather_sun', 'weather_cloud', 'weather_rain',
    'swatch_edge_1', 'swatch_edge_2', 'swatch_edge_3',
    'swatch_print_1', 'swatch_print_2', 'swatch_print_3',
    'txt_normal', 'txt_crowded', 'txt_covered',
    'route_up', 'route_down', 'route_left', 'route_right'
]

EDGE_CLS = ('edge_type_', 'print_edge_')                  # box cạnh (kiểu nét) -> status lấy từ legend
EDGE_BOX_CLS = {'closed'}                                 # closed = box phủ cả đoạn đường
MODIFIER_CLS = {'stairs', 'oneway_up', 'oneway_down', 'oneway_left', 'oneway_right'}  # icon nhỏ giữa cạnh
NODE_CLS = set(CLASSES[:14])                              # landmark + robot


# ==========================================
# HELPERS
# ==========================================
def get_center(box):
    return ((box[0] + box[2]) / 2, (box[1] + box[3]) / 2)


def get_distance(p1, p2):
    return math.sqrt((p1[0] - p2[0]) ** 2 + (p1[1] - p2[1]) ** 2)


def dynamic_cluster(coords, unit_size):
    if not coords:
        return []
    coords = sorted(coords)
    clusters = [[coords[0]]]
    tolerance = unit_size * 0.4
    for c in coords[1:]:
        if c - np.mean(clusters[-1]) <= tolerance:
            clusters[-1].append(c)
        else:
            clusters.append([c])
    return [float(np.mean(cl)) for cl in clusters]


def get_grid_index(val, clusters):
    return min(range(len(clusters)), key=lambda i: abs(clusters[i] - val))


def is_plain_edge(c):
    return c.startswith(EDGE_CLS)


# ==========================================
# LEGEND: swatch k -> txt bên cạnh -> status
# ==========================================
def build_legend(dets, txt_side='right', pairs_out=None):
    texts = [d for d in dets if d['class'].startswith('txt_')]
    swatches = [d for d in dets if d['class'].startswith(('swatch_edge_', 'swatch_print_'))]
    if not texts or not swatches:
        return {}

    def find_label(s):
        sx1, _, sx2, _ = s['box']
        slack = 0.3 * s['w']
        row = [t for t in texts
               if abs(t['center'][1] - s['center'][1]) <= max(s['h'], t['h']) * 0.6]
        if txt_side == 'left':
            side = [t for t in row if t['box'][2] <= sx1 + slack]
        else:
            side = [t for t in row if t['box'][0] >= sx2 - slack]
        pool = side or row or texts
        return min(pool, key=lambda t: get_distance(t['center'], s['center']))

    best = {}  # k -> (conf swatch, status)
    for s in swatches:
        k = s['class'][-1]
        t = find_label(s)
        if pairs_out is not None:
            pairs_out.append((s, t))
        status = t['class'].replace('txt_', '')
        if k not in best or s['conf'] > best[k][0]:
            best[k] = (s['conf'], status)

    legend = {}
    for k, (_, status) in best.items():
        legend['edge_type_' + k] = status
        legend['print_edge_' + k] = status
    return legend


def _oneway_target(direction, na, nb):
    if direction == 'up':
        return [min(na[0], nb[0]), na[1]]
    if direction == 'down':
        return [max(na[0], nb[0]), na[1]]
    if direction == 'left':
        return [na[0], min(na[1], nb[1])]
    return [na[0], max(na[1], nb[1])]  # right


# ==========================================
# 1. YOLO -> DETS (chạy model 1 lần, lọc ngưỡng sau)
# ==========================================
def predict_dets(model, image_path, min_conf=0.1):
    res = model.predict(image_path, conf=min_conf, verbose=False)[0]
    img = res.orig_img[:, :, ::-1]  # BGR -> RGB
    dets = []
    for box, cls_id, cf in zip(res.boxes.xyxy.cpu().numpy(), res.boxes.cls.cpu().numpy(),
                               res.boxes.conf.cpu().numpy()):
        box = [float(v) for v in box]
        dets.append({'class': CLASSES[int(cls_id)], 'box': box, 'conf': float(cf),
                     'center': get_center(box), 'w': box[2] - box[0], 'h': box[3] - box[1]})
    return dets, img


def collect_dets(model, scenes, img_dir, min_conf=0.1):
    """Chạy YOLO trên toàn bộ val 1 lần, trả {scene_id: dets}. Sau đó thử ngưỡng khác không cần chạy lại model."""
    cache = {}
    for sc in tqdm(scenes, desc="YOLO predict"):
        path = os.path.join(img_dir, f"{sc['scene_id']}.jpg")
        if os.path.exists(path):
            cache[sc['scene_id']] = predict_dets(model, path, min_conf)[0]
    return cache


# ==========================================
# 2. DETS -> JSON GRAPH
#    box cạnh   : edge_type_k / print_edge_k (status = legend) / closed
#    icon nhỏ   : stairs / oneway_*  -> gán vào cạnh có box chứa tâm icon
#    split_long : box cạnh dài >1 ô -> tách thành nhiều cạnh 1 ô (False = giữ kiểu cũ: 1 cạnh nối xa)
# ==========================================
def dets_to_scene(dets, mission_via=None, mission_goal=None, conf=0.4, edge_conf=None,
                  txt_side='right', split_long=True, trace=None, route_conf=None,
                  mission_via_rc=None, mission_goal_rc=None):
    edge_conf = conf if edge_conf is None else edge_conf
    route_conf = conf if route_conf is None else route_conf
    dets = [dict(d) for d in dets
            if d['conf'] >= (edge_conf if is_plain_edge(d['class']) else route_conf if d['class'].startswith('route_') else conf)]

    pairs = []
    legend = build_legend(dets, txt_side=txt_side, pairs_out=pairs)
    for d in dets:
        if is_plain_edge(d['class']):
            d['status'] = legend.get(d['class'], '?')

    edge_dets = [d for d in dets if is_plain_edge(d['class']) or d['class'] in EDGE_BOX_CLS]
    modifiers = [d for d in dets if d['class'] in MODIFIER_CLS]

    # thước đo ô lưới: chỉ đo trên box cạnh (không tính icon)
    hw = [d['w'] for d in edge_dets if d['w'] > d['h']]
    vh = [d['h'] for d in edge_dets if d['h'] > d['w']]
    grid_w = float(np.median(hw)) if hw else 100.0
    grid_h = float(np.median(vh)) if vh else 100.0

    scene = {'nodes': [], 'edges': [], 'landmarks': [], 'robot': {},
             'weather': 'rain' if any(d['class'] == 'weather_rain' for d in dets) else 'dry',
             'mission': {'via': mission_via, 'goal': mission_goal}, 'warnings': []}

    # --- lưới động
    raw_x, raw_y = [], []
    for d in dets:
        if d['class'] in NODE_CLS:
            raw_x.append(d['center'][0])
            raw_y.append(d['center'][1])
    for d in edge_dets:
        x1, y1, x2, y2 = d['box']
        if d['w'] >= d['h']:
            raw_y.append(d['center'][1])
            raw_x += [x1, x2]
        else:
            raw_x.append(d['center'][0])
            raw_y += [y1, y2]
    col_cl, row_cl = dynamic_cluster(raw_x, grid_w), dynamic_cluster(raw_y, grid_h)

    # --- landmark / robot
    for d in dets:
        if d['class'] not in NODE_CLS:
            continue
        r, c = get_grid_index(d['center'][1], row_cl), get_grid_index(d['center'][0], col_cl)
        if d['class'].startswith('robot_'):
            scene['robot'] = {'rc': [r, c], 'heading': d['class'].split('_')[1].upper()}
        else:
            scene['landmarks'].append({'type': d['class'], 'rc': [r, c]})
    for r in range(len(row_cl)):
        for c in range(len(col_cl)):
            scene['nodes'].append({'rc': [r, c]})

    # --- cạnh
    edges = {}
    for d in edge_dets:
        x1, y1, x2, y2 = d['box']
        horiz = d['w'] >= d['h']
        if horiz:
            r = get_grid_index(d['center'][1], row_cl)
            ca, cb = sorted([get_grid_index(x1, col_cl), get_grid_index(x2, col_cl)])
            steps = [([r, c], [r, c + 1]) for c in range(ca, cb)]
        else:
            c = get_grid_index(d['center'][0], col_cl)
            ra, rb = sorted([get_grid_index(y1, row_cl), get_grid_index(y2, row_cl)])
            steps = [([r, c], [r + 1, c]) for r in range(ra, rb)]
        if not steps:
            continue
        if len(steps) > 1:
            scene['warnings'].append(f"{d['class']} dai {len(steps)} o: {steps[0][0]}->{steps[-1][1]}"
                                     f" ({'tach' if split_long else 'giu nguyen'})")
            if not split_long:
                steps = [(steps[0][0], steps[-1][1])]

        for na, nb in steps:
            if horiz:
                box = [col_cl[na[1]], y1, col_cl[nb[1]], y2]
            else:
                box = [x1, row_cl[na[0]], x2, row_cl[nb[0]]]
            key = (tuple(na), tuple(nb))
            e = edges.setdefault(key, {'a': na, 'b': nb, 'status': None, 'stairs': False,
                                       'oneway_to': None, 'route_status': None, 'route_statuses': [], 'route_to': None, '_box': box, '_horiz': horiz,
                                       '_center': get_center(box), '_plain_conf': -1.0,
                                       '_closed': False, '_src': None})
            cls = d['class']
            if is_plain_edge(cls):
                if d['conf'] > e['_plain_conf']:      # 2 box plain chồng nhau -> lấy box conf cao hơn
                    e['_plain_conf'] = d['conf']
                    e['status'] = legend.get(cls, 'normal')
                    e['_src'] = ('legend:' if cls in legend else 'NO_LEGEND:') + cls
            else:  # closed
                e['_closed'] = True

    for e in edges.values():
        if e['_closed']:
            e['status'] = 'closed'
            e['_src'] = 'closed_box'

    # --- icon nhỏ -> cạnh có box chứa tâm icon, đúng hướng
    tol = 0.15 * min(grid_w, grid_h)
    for m in modifiers:
        cx, cy = m['center']
        direction = m['class'].split('_')[1] if m['class'].startswith('oneway_') else None
        best, best_d = None, None
        for e in edges.values():
            x1, y1, x2, y2 = e['_box']
            if not (x1 - tol <= cx <= x2 + tol and y1 - tol <= cy <= y2 + tol):
                continue
            if direction in ('left', 'right') and not e['_horiz']:
                continue
            if direction in ('up', 'down') and e['_horiz']:
                continue
            dist = get_distance(m['center'], e['_center'])
            if best is None or dist < best_d:
                best, best_d = e, dist
        if best is None:
            scene['warnings'].append(f"icon {m['class']} tai {np.round(m['center']).tolist()} khong nam trong canh nao")
            continue
        if m['class'] == 'stairs':
            best['stairs'] = True
        else:
            best['oneway_to'] = _oneway_target(direction, best['a'], best['b'])

    # Route classes are a second edge status; keep road status unchanged.
    route_dirs={'route_up':(-1,0),'route_down':(1,0),'route_left':(0,-1),'route_right':(0,1)}
    route_edges=[]
    for rd in [d for d in dets if d['class'] in route_dirs]:
        dr,dc=route_dirs[rd['class']]
        best=None; best_dist=float('inf')
        for e in edges.values():
            a,b=tuple(e['a']),tuple(e['b'])
            if (b[0]-a[0],b[1]-a[1])==(dr,dc): src,dst=a,b
            elif (a[0]-b[0],a[1]-b[1])==(dr,dc): src,dst=b,a
            else: continue
            p0=np.array([col_cl[src[1]],row_cl[src[0]]],float)
            p1=np.array([col_cl[dst[1]],row_cl[dst[0]]],float)
            vec=p1-p0; L=float(np.linalg.norm(vec))
            if L<2: continue
            u=vec/L; right=np.array([-u[1],u[0]])
            expected=p0+ROUTE_CENTER_T*vec+ROUTE_LANE_OFFSET_RATIO*L*right
            # Normalize by cell pitch so skewed/hard maps share a scale.
            pitch=max(1.0,min(grid_w,grid_h))
            dist=float(np.linalg.norm(np.asarray(rd['center'],float)-expected))/pitch
            if dist<best_dist: best_dist=dist; best=(src,dst,e)
        if best is None or best_dist>0.45:
            scene['warnings'].append(f"{rd['class']} không ghép được cạnh route (normalized_error={best_dist:.3f})")
            continue
        src,dst,e=best; direction=rd['class'].replace('route_','')
        e['route_statuses'].append(rd['class'])
        # route_status is separate from road status and supports both directions on one road.
        unique=sorted(set(e['route_statuses']))
        e['route_statuses']=unique
        e['route_status']=unique[0] if len(unique)==1 else 'multiple'
        e['route_to']=list(dst) if len(unique)==1 else None
        route_edges.append({'from':list(src),'to':list(dst),'direction':direction,
                            'class':rd['class'],'confidence':float(rd['conf']),
                            'offset_error_cells':best_dist,'edge_status':e.get('status')})
    scene['route_edges']=route_edges
    goal_ref=scene.get('mission',{}).get('goal')
    via_ref=scene.get('mission',{}).get('via')
    goal_targets=[tuple(mission_goal_rc)] if mission_goal_rc is not None else [tuple(l['rc']) for l in scene['landmarks'] if l['type']==mission_goal]
    via_targets=[tuple(mission_via_rc)] if mission_via_rc is not None else [tuple(l['rc']) for l in scene['landmarks'] if l['type']==mission_via]
    if mission_via is None: via_targets=[]
    rec=reconstruct_route(route_edges,scene.get('robot',{}).get('rc'),goal_targets,via_targets,max_steps=45)
    scene['route_moves']=rec['moves']
    scene['route_nodes']=rec['nodes']
    scene['route_reconstruction']={'ok':rec['ok'],'ambiguous':rec['ambiguous'],
                                   'used_edges':rec['used_edges'],'detected_edges':len(route_edges),
                                   'reason':rec['reason']}
    if not rec['ok']:
        scene['warnings'].append('Không ráp được route đầy đủ: '+rec['reason'])
    if trace is not None:
        trace['route_edges']=route_edges
        trace['route_reconstruction']=scene['route_reconstruction']
    for e in edges.values():
        e['route_statuses']=sorted(set(e.get('route_statuses',[])))
        for k in ('_box', '_horiz', '_center', '_plain_conf', '_closed', '_src'):
            e.pop(k, None)
        scene['edges'].append(e)
    return scene


# Thay toàn bộ reconstruct_route hiện tại bằng hàm này
import heapq
import math

def reconstruct_route(
    route_edges,
    start,
    goal_targets,
    via_targets=None,
    max_steps=45,
    beam_size=None,  # giữ tham số để tương thích với các chỗ gọi cũ
):
    """Tìm route chi phí thấp nhất từ robot → via (nếu có) → goal."""
    if start is None:
        return {
            "ok": False, "moves": [], "nodes": [],
            "ambiguous": False, "used_edges": 0,
            "reason": "missing_robot",
        }

    start = tuple(int(x) for x in start)
    goals = {tuple(int(x) for x in x) for x in (goal_targets or [])}
    vias = {tuple(int(x) for x in x) for x in (via_targets or [])}

    if not route_edges:
        return {
            "ok": False, "moves": [], "nodes": [list(start)],
            "ambiguous": False, "used_edges": 0,
            "reason": "no_route_detections",
        }

    if not goals:
        return {
            "ok": False, "moves": [], "nodes": [list(start)],
            "ambiguous": False, "used_edges": 0,
            "reason": "goal_not_resolved",
        }

    # Nối các phát hiện cùng cạnh có hướng; giữ phát hiện có chi phí thấp nhất.
    # Tọa độ quyết định hướng bước, tránh phụ thuộc chuỗi direction.
    direction_to_move = {
        (-1, 0): 0,  # UP
        (1, 0): 1,   # DOWN
        (0, -1): 2,  # LEFT
        (0, 1): 3,   # RIGHT
    }

    edge_by_key = {}
    for edge in route_edges:
        try:
            a = tuple(int(x) for x in edge["from"])
            b = tuple(int(x) for x in edge["to"])
        except (KeyError, TypeError, ValueError):
            continue

        delta = (b[0] - a[0], b[1] - a[1])
        if delta not in direction_to_move:
            continue

        confidence = max(
            1e-5,
            min(1.0, float(edge.get("confidence", 0.5))),
        )
        offset_error = max(
            0.0,
            min(2.0, float(edge.get("offset_error_cells", 0.0))),
        )

        # Mỗi bước có chi phí cơ bản 1.0 để tránh route vòng.
        # Confidence và sai lệch hình học chỉ phân biệt các route gần ngang nhau.
        edge_cost = (
            1.0
            + 0.25 * (1.0 - confidence)
            + 0.25 * offset_error
        )

        key = (a, b)
        old = edge_by_key.get(key)
        if old is None or edge_cost < old["cost"]:
            edge_by_key[key] = {
                "from": a,
                "to": b,
                "move": direction_to_move[delta],
                "cost": edge_cost,
            }

    adjacency = {}
    for edge in edge_by_key.values():
        adjacency.setdefault(edge["from"], []).append(edge)

    initial_via_seen = (not vias) or (start in vias)
    start_state = (start, initial_via_seen)

    # Dijkstra trên trạng thái (node, đã ghé via chưa).
    best = {start_state: (0.0, 0)}
    parent = {}
    heap = [(0.0, 0, start, initial_via_seen)]
    final_state = None

    while heap:
        cost, steps, node, via_seen = heapq.heappop(heap)
        state = (node, via_seen)

        if (cost, steps) != best.get(state):
            continue

        # Dừng tại goal đầu tiên sau khi đã ghé via.
        if node in goals and via_seen and steps > 0:
            final_state = state
            break

        if steps >= max_steps:
            continue

        for edge in adjacency.get(node, []):
            nxt = edge["to"]
            next_via_seen = via_seen or (nxt in vias)
            next_state = (nxt, next_via_seen)

            new_cost = cost + edge["cost"]
            new_steps = steps + 1
            candidate = (new_cost, new_steps)

            if candidate < best.get(next_state, (math.inf, math.inf)):
                best[next_state] = candidate
                parent[next_state] = (state, edge)
                heapq.heappush(
                    heap,
                    (new_cost, new_steps, nxt, next_via_seen),
                )

    if final_state is None:
        return {
            "ok": False,
            "moves": [],
            "nodes": [list(start)],
            "ambiguous": False,
            "used_edges": 0,
            "reason": "no directed route from robot through via to goal",
        }

    # Lần ngược trạng thái để dựng route.
    moves_reversed = []
    nodes_reversed = [final_state[0]]
    used_edges = set()
    state = final_state

    while state != start_state:
        if state not in parent:
            return {
                "ok": False, "moves": [], "nodes": [list(start)],
                "ambiguous": False, "used_edges": 0,
                "reason": "broken_parent_chain",
            }

        previous_state, edge = parent[state]
        moves_reversed.append(edge["move"])
        nodes_reversed.append(previous_state[0])
        used_edges.add((edge["from"], edge["to"]))
        state = previous_state

    moves = list(reversed(moves_reversed))
    nodes = [list(node) for node in reversed(nodes_reversed)]

    return {
        "ok": True,
        "moves": moves,
        "nodes": nodes,
        "ambiguous": False,
        "used_edges": len(used_edges),
        "reason": "ok",
    }


def yolo_to_scene_json(image_path, model, mission_via=None, mission_goal=None, conf=0.4,
                       edge_conf=None, split_long=True, txt_side='right', trace=None, route_conf=None,
                       mission_via_rc=None, mission_goal_rc=None):
    model = YOLO(model) if isinstance(model, str) else model
    dets, img = predict_dets(model, image_path, min_conf=min(conf, edge_conf or conf, route_conf or conf))
    scene = dets_to_scene(dets, mission_via, mission_goal, conf, edge_conf, txt_side, split_long, trace,
                          route_conf=route_conf, mission_via_rc=mission_via_rc, mission_goal_rc=mission_goal_rc)
    if trace is not None:
        trace['img'] = img
    return scene


# ==========================================
# 3. SO SÁNH VỚI GROUND TRUTH
# ==========================================
def _edge_dict(scene):
    out = {}
    for e in scene.get('edges', []):
        k = tuple(sorted([tuple(e['a']), tuple(e['b'])]))
        ow = tuple(e['oneway_to']) if e.get('oneway_to') else None
        out[k] = {'status': e.get('status', 'normal'), 'stairs': bool(e.get('stairs')), 'oneway_to': ow}
    return out


def compare_scenes(pred, true, edge_src=None):
    """Trả (errors, counts). errors: list dict {type, key, pred, true, src}."""
    edge_src = edge_src or {}
    errors = []
    P, T = _edge_dict(pred), _edge_dict(true)

    def src_of(k):
        # edge_src key = (a, b) đã sort sẵn
        return (edge_src.get(k) or '-').split(':')[0]

    n_correct = 0
    for k in sorted(set(P) | set(T)):
        p, t = P.get(k), T.get(k)
        if p is None:
            errors.append({'type': 'edge_missing', 'key': k, 'pred': None, 'true': t, 'src': '-'})
        elif t is None:
            errors.append({'type': 'edge_extra', 'key': k, 'pred': p, 'true': None, 'src': src_of(k)})
        else:
            ok = True
            for field, etype in (('status', 'edge_wrong_status'), ('stairs', 'edge_wrong_stairs'),
                                 ('oneway_to', 'edge_wrong_oneway_to')):
                if p[field] != t[field]:
                    ok = False
                    errors.append({'type': etype, 'key': k, 'pred': p[field], 'true': t[field],
                                   'src': src_of(k)})
            n_correct += ok

    lm_p = {(lm['type'], tuple(lm['rc'])) for lm in pred.get('landmarks', [])}
    lm_t = {(lm['type'], tuple(lm['rc'])) for lm in true.get('landmarks', [])}
    for x in sorted(lm_t - lm_p):
        errors.append({'type': 'landmark_missing', 'key': x, 'pred': None, 'true': x, 'src': '-'})
    for x in sorted(lm_p - lm_t):
        errors.append({'type': 'landmark_extra', 'key': x, 'pred': x, 'true': None, 'src': '-'})

    def rob(sc):
        r = sc.get('robot') or {}
        return (tuple(r['rc']), r.get('heading', '').upper()) if r.get('rc') else None
    if rob(pred) != rob(true):
        errors.append({'type': 'robot_wrong', 'key': 'robot', 'pred': rob(pred), 'true': rob(true), 'src': '-'})

    def wea(sc):
        return 'dry' if sc.get('weather', 'dry') == 'dry' else 'rain'
    if wea(pred) != wea(true):
        errors.append({'type': 'weather_wrong', 'key': 'weather', 'pred': wea(pred), 'true': wea(true), 'src': '-'})

    return errors, {'gt_edges': len(T), 'pred_edges': len(P), 'edge_correct': n_correct}


# ==========================================
# 4. QUÉT TOÀN BỘ VAL
# ==========================================
def evaluate_val(scenes, dets_cache, conf=0.4, edge_conf=None, split_long=True, txt_side='right',
                 quiet=False, max_list=80, save_path=None):
    n_img = n_exact = 0
    tot = Counter()
    err_types = Counter()
    status_conf = Counter()      # (pred -> true) cho edge_wrong_status
    status_src = Counter()       # nguồn status của cạnh bị sai status
    failures = {}

    for sc in scenes:
        sid = sc['scene_id']
        if sid not in dets_cache:
            continue
        trace = {}
        pred = dets_to_scene(dets_cache[sid], sc['mission'].get('via'), sc['mission'].get('goal'),
                             conf=conf, edge_conf=edge_conf, txt_side=txt_side,
                             split_long=split_long, trace=trace)
        errs, cnt = compare_scenes(pred, sc, trace['edge_src'])
        n_img += 1
        tot.update(cnt)
        if not errs:
            n_exact += 1
            continue
        for e in errs:
            err_types[e['type']] += 1
            if e['type'] == 'edge_wrong_status':
                status_conf[(e['pred'], e['true'])] += 1
                status_src[e['src']] += 1
        failures[sid] = [{**e, 'key': str(e['key']), 'pred': str(e['pred']), 'true': str(e['true'])}
                         for e in errs]

    rec = tot['edge_correct'] / max(tot['gt_edges'], 1)
    prec = tot['edge_correct'] / max(tot['pred_edges'], 1)
    summary = {'images': n_img, 'exact': n_exact, 'exact_rate': n_exact / max(n_img, 1),
               'edge_recall': rec, 'edge_precision': prec, 'errors': dict(err_types)}

    if not quiet:
        print("=" * 64)
        print(f"conf={conf} edge_conf={edge_conf} split_long={split_long}")
        print(f"DUNG HOAN TOAN : {n_exact}/{n_img} anh ({100 * n_exact / max(n_img, 1):.1f}%)  | SAI: {n_img - n_exact} anh")
        print(f"Canh dung nguyen ven: {tot['edge_correct']}/{tot['gt_edges']} GT "
              f"(recall {100 * rec:.2f}%) | {tot['pred_edges']} canh du doan (precision {100 * prec:.2f}%)")
        print("\nLoai loi:")
        for k, v in err_types.most_common():
            print(f"  {k:<24}{v}")
        if status_conf:
            print("\nedge_wrong_status  (pred -> true):")
            for (p, t), v in status_conf.most_common():
                print(f"  {p:>8} -> {t:<8} {v}")
            print("nguon status cua cac canh sai (NO_LEGEND = thieu legend, legend = model phan loai sai net):")
            for k, v in status_src.most_common():
                print(f"  {k:<14}{v}")
        print("\nDANH SACH ANH SAI:")
        for sid in sorted(failures)[:max_list]:
            print(f"[{sid}] ({len(failures[sid])} loi)")
            for e in failures[sid]:
                print(f"   - {e['type']}: {e['key']} pred={e['pred']} true={e['true']}")
        if len(failures) > max_list:
            print(f"... con {len(failures) - max_list} anh nua (xem file json)")
    if save_path:
        with open(save_path, 'w', encoding='utf-8') as f:
            json.dump(failures, f, ensure_ascii=False, indent=1)
        if not quiet:
            print(f"\nDa luu chi tiet vao {save_path}")
    return summary


def sweep(scenes, dets_cache, configs):
    """configs: list (ten, dict kwargs cho evaluate_val). Chi hau xu ly, khong chay lai YOLO."""
    print(f"{'config':<28}{'exact':>10}{'recall':>9}{'prec':>9}   loi")
    for name, kw in configs:
        s = evaluate_val(scenes, dets_cache, quiet=True, **kw)
        n_err = sum(s['errors'].values())
        print(f"{name:<28}{s['exact']:>4}/{s['images']:<5}{100 * s['edge_recall']:>8.2f}%"
              f"{100 * s['edge_precision']:>8.2f}%   {n_err}")


# ==========================================
# 5. VẼ (debug 1 ảnh)
# ==========================================
def show_labeled_detections(trace, figsize=(18, 18)):
    COL = {'normal': 'limegreen', 'crowded': 'red', 'covered': 'blue', '?': 'magenta'}
    fig, ax = plt.subplots(figsize=figsize)
    ax.imshow(trace['img'])
    for d in trace['dets']:
        cls = d['class']
        x1, y1, x2, y2 = d['box']
        if is_plain_edge(cls):
            st = d.get('status', '?')
            color = COL.get(st, 'magenta')
            tag = cls.replace('print_edge_', 'p').replace('edge_type_', 'e') + '=' + st
            tx, ty, va = (x1 + x2) / 2, (y1 + y2) / 2, 'center'
        elif cls.startswith(('swatch_', 'txt_')):
            color, tag, tx, ty, va = 'red', f"{cls} {d['conf']:.2f}", x1, y1 - 2, 'bottom'
        elif cls in MODIFIER_CLS or cls == 'closed':
            color, tag, tx, ty, va = 'darkorange', cls, x1, y1 - 2, 'bottom'
        elif cls.startswith('weather_'):
            color, tag, tx, ty, va = 'gray', cls, x1, y1 - 2, 'bottom'
        else:
            color, tag, tx, ty, va = 'cyan', cls, x1, y1 - 2, 'bottom'
        ax.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor=color, linewidth=1.3))
        ax.text(tx, ty, tag, color=color, fontsize=7, ha='center' if va == 'center' else 'left', va=va,
                fontweight='bold', bbox=dict(facecolor='white', alpha=0.65, pad=0.5, edgecolor='none'))
    for s, t in trace['pairs']:
        ax.annotate('', xy=t['center'], xytext=s['center'], arrowprops=dict(arrowstyle='->', color='lime', lw=2))
    ax.set_title("DETECTION + LABEL (e/p = edge_type/print_edge, sau dau '=' la status theo legend)")
    ax.axis('off')
    plt.tight_layout()
    plt.show()


def draw_scene_map(scene, title="Scene Map", ax=None, highlight=None):
    own = ax is None
    if own:
        fig, ax = plt.subplots(figsize=(10, 10))
    ax.set_title(title, fontsize=14, fontweight='bold')
    highlight = highlight or set()

    def get_xy(rc):
        return rc[1], -rc[0]

    color_map = {'normal': 'black', 'crowded': 'red', 'covered': 'blue', 'closed': 'gray'}
    style_map = {'normal': '-', 'crowded': '--', 'covered': ':', 'closed': '-.'}

    for edge in scene.get('edges', []):
        x1, y1 = get_xy(edge['a'])
        x2, y2 = get_xy(edge['b'])
        st = edge.get('status', 'normal')
        ax.plot([x1, x2], [y1, y2], color=color_map.get(st, 'black'),
                linestyle=style_map.get(st, '-'), linewidth=3, alpha=0.7, zorder=1)
        mx, my = (x1 + x2) / 2, (y1 + y2) / 2
        key = tuple(sorted([tuple(edge['a']), tuple(edge['b'])]))
        if key in highlight:
            ax.plot([x1, x2], [y1, y2], color='magenta', linewidth=11, alpha=0.3, zorder=0)
            ax.text(mx, my + 0.12, st, color='magenta', fontsize=9, ha='center', fontweight='bold', zorder=6)
        if st == 'closed':
            ax.text(mx, my, 'X', color='darkred', fontsize=16, fontweight='bold', ha='center', va='center', zorder=3)
        if edge.get('stairs'):
            ax.text(mx, my, '|||', color='orange', fontsize=14, fontweight='bold', ha='center', va='center', zorder=3)
        if edge.get('oneway_to'):
            tx, ty = get_xy(edge['oneway_to'])
            other = edge['a'] if tuple(edge['oneway_to']) == tuple(edge['b']) else edge['b']
            ox, oy = get_xy(other)
            ux, uy = (tx - ox), (ty - oy)
            ax.annotate('', xy=(mx + ux * 0.2, my + uy * 0.2), xytext=(mx - ux * 0.2, my - uy * 0.2),
                        arrowprops=dict(arrowstyle='-|>', color='green', lw=3), zorder=4)

    for lm in scene.get('landmarks', []):
        x, y = get_xy(lm['rc'])
        ax.scatter(x, y, s=600, c='lightgreen', edgecolors='green', zorder=4)
        ax.text(x, y + 0.25, lm['type'], fontsize=11, ha='center', fontweight='bold', zorder=5)

    if scene.get('robot'):
        rx, ry = get_xy(scene['robot']['rc'])
        heading = scene['robot'].get('heading', 'UP')
        dx, dy = {'UP': (0, 0.4), 'DOWN': (0, -0.4), 'LEFT': (-0.4, 0), 'RIGHT': (0.4, 0)}.get(heading, (0, 0))
        ax.scatter(rx, ry, s=800, c='cyan', edgecolors='blue', marker='o', zorder=6)
        ax.arrow(rx, ry, dx, dy, head_width=0.2, head_length=0.2, fc='blue', ec='blue', zorder=7)
        ax.text(rx, ry - 0.3, "ROBOT", fontsize=11, color='blue', ha='center', fontweight='bold')

    ax.grid(True, linestyle=':', alpha=0.6)
    ax.set_aspect('equal')
    if own:
        plt.show()


def diagnose(image_path, model, true_scene, conf=0.4, edge_conf=None, split_long=True,
             txt_side='right', show=True):
    trace = {}
    pred = yolo_to_scene_json(image_path, model, true_scene['mission'].get('via'),
                              true_scene['mission'].get('goal'), conf=conf, edge_conf=edge_conf,
                              split_long=split_long, txt_side=txt_side, trace=trace)
    errs, _ = compare_scenes(pred, true_scene, trace['edge_src'])
    print("=" * 70)
    print(image_path)
    print("legend:", trace['legend'])
    for s, t in trace['pairs']:
        print(f"  {s['class']} ({s['conf']:.2f}) -> {t['class']} ({t['conf']:.2f})")
    for e in errs:
        print(f"  {e['type']:<22}{e['key']} pred={e['pred']} true={e['true']} src={e['src']}")
    if not errs:
        print("  => DUNG HOAN TOAN")
    if pred['warnings']:
        print("warnings:", pred['warnings'])
    if show:
        wrong = {e['key'] for e in errs if isinstance(e['key'], tuple) and len(e['key']) == 2
                 and isinstance(e['key'][0], tuple)}
        show_labeled_detections(trace)
        fig, axs = plt.subplots(1, 2, figsize=(20, 10))
        draw_scene_map(true_scene, "GROUND TRUTH", ax=axs[0], highlight=wrong)
        draw_scene_map(pred, "DU DOAN (hong = canh sai)", ax=axs[1], highlight=wrong)
        plt.tight_layout()
        plt.show()
    return pred, trace, errs




In [ ]:
# CELL 2 — Bộ mô phỏng và sweep policy (đầy đủ, không cần import file ngoài)
#!/usr/bin/env python3
"""Exact strategy-family sweep for the Campus Courier route task.

One hidden policy is shared by all demos and queries in an episode.  This
module enumerates the public policy family, simulates demo routes, filters the
policies that exactly reproduce every observed demo, then predicts queries.

Scene inputs use the annotated `scenes.json` schema.  For test inference, a
scene parser should provide the same graph/mission fields and put the YOLO-read
demo route in scene['route']['moves'] (or episode demo['route']). Spatial
references should already be grounded to `ref['rc']` by the map/NLU layer.
"""
from __future__ import annotations

import argparse
import heapq
import itertools
import json
import tempfile
import zipfile
from collections import defaultdict
from pathlib import Path
from typing import Any, Iterable


DIRECTIONS = {0: (-1, 0), 1: (1, 0), 2: (0, -1), 3: (0, 1)}
HEADING_TO_DIR = {"UP": 0, "DOWN": 1, "LEFT": 2, "RIGHT": 3}
TURN_TABLE = {
    "none": (0, 0, 0, 0),
    "straight": (0, 30, 30, 300),
    "right": (0, 5, 45, 90),
    "left": (0, 45, 5, 90),
}
SMOOTH_TURNS = (0, 15, 15, 150)

# The 4,608 public configurations: 2*4*4*3*3*4*4.
FAMILY = {
    "legged": (False, True),
    "crowded": (10, 20, 35, 60),
    "covered": (5, 7, 10, 14),
    "rain": ("none", "cover", "stairs"),
    "urgent": ("none", "direct", "bold"),
    "fragile": ("none", "calm", "smooth", "both"),
    "turns": ("none", "straight", "right", "left"),
}
PARAMETER_ORDER = tuple(FAMILY)


def all_strategies() -> Iterable[dict[str, Any]]:
    """Yield every strategy in stable order."""
    names = PARAMETER_ORDER
    for values in itertools.product(*(FAMILY[n] for n in names)):
        yield dict(zip(names, values))


def _rc(value: Any) -> tuple[int, int]:
    return tuple(value)  # type: ignore[return-value]


def _step_dir(a: tuple[int, int], b: tuple[int, int]) -> int:
    delta = (b[0] - a[0], b[1] - a[1])
    for direction, offset in DIRECTIONS.items():
        if offset == delta:
            return direction
    raise ValueError(f"Non-adjacent map nodes in edge: {a} -> {b}")


def _turn_kind(old: int, new: int) -> int:
    """Index 0=straight, 1=right, 2=left, 3=reverse (screen coordinates)."""
    if old == new:
        return 0
    right = {0: 3, 3: 1, 1: 2, 2: 0}
    left = {0: 2, 2: 1, 1: 3, 3: 0}
    if new == right[old]:
        return 1
    if new == left[old]:
        return 2
    return 3


def _tie_rank(heading: int, direction: int) -> int:
    """Public COURIER tie-break order: straight, right, left, reverse."""
    return _turn_kind(heading, direction)


def _is_urgent(mission: dict[str, Any]) -> bool:
    return bool(mission.get("urgent", False))


def _is_fragile(mission: dict[str, Any]) -> bool:
    return bool(mission.get("fragile", False))


def _edge_cost(edge: dict[str, Any], strategy: dict[str, Any], mission: dict[str, Any], weather: str) -> int | None:
    """Apply edge-cost overrides in the exact order from the statement."""
    status = edge.get("status", "normal")
    if status == "closed":
        return None

    # direct+urgent short-circuits every cost modifier, but not stair access.
    if _is_urgent(mission) and strategy["urgent"] == "direct":
        return 10

    cost = strategy["crowded"] if status == "crowded" else strategy["covered"] if status == "covered" else 10

    # Ordered overrides: bold, calm/both, rain cover, stair surcharge.
    if strategy["urgent"] == "bold" and _is_urgent(mission) and status == "crowded":
        cost = 10
    if strategy["fragile"] in ("calm", "both") and _is_fragile(mission) and status == "crowded":
        cost *= 3
    if strategy["rain"] == "cover" and weather == "rain" and status != "covered":
        cost *= 3
    if edge.get("stairs", False) and strategy["rain"] == "stairs" and weather == "rain":
        cost += 40
    return cost


def _turn_cost(strategy: dict[str, Any], mission: dict[str, Any], old: int, new: int) -> int:
    row = TURN_TABLE[strategy["turns"]]
    if _is_fragile(mission) and strategy["fragile"] in ("smooth", "both"):
        row = tuple(max(a, b) for a, b in zip(row, SMOOTH_TURNS))
    return row[_turn_kind(old, new)]


def _target_nodes(scene: dict[str, Any], kind: str | None, ref_key: str) -> set[tuple[int, int]]:
    if not kind:
        return set()
    mission = scene["mission"]
    ref = mission.get(ref_key) or {}
    # The NLP/map layer should resolve north/south/east/west/near/far to rc.
    if ref.get("rc") is not None:
        return {_rc(ref["rc"])}
    copies = {_rc(item["rc"]) for item in scene.get("landmarks", []) if item.get("type") == kind}
    if not copies:
        raise ValueError(f"No landmark {kind!r} in scene {scene.get('scene_id')}")
    if len(copies) == 1:
        return copies
    if ref.get("kind") not in (None, "", "any"):
        raise ValueError(
            f"Spatial mission reference {ref!r} needs an rc before solving scene {scene.get('scene_id')}"
        )
    return copies


def solve_route(scene: dict[str, Any], strategy: dict[str, Any]) -> list[int] | None:
    """Return the deterministic minimum-cost absolute route, or None if unreachable."""
    mission = scene["mission"]
    weather = scene.get("weather", "dry")
    nodes = {_rc(node["rc"]) for node in scene.get("nodes", [])}
    source = _rc(scene["robot"]["rc"])
    heading = HEADING_TO_DIR[str(scene["robot"]["heading"]).upper()]
    goal_nodes = _target_nodes(scene, mission.get("goal"), "goal_ref")
    via_kind = mission.get("via")
    via_nodes = _target_nodes(scene, via_kind, "via_ref") if via_kind else set()
    if not goal_nodes:
        raise ValueError(f"No goal for scene {scene.get('scene_id')}")
    if via_kind and not via_nodes:
        raise ValueError(f"No via target for scene {scene.get('scene_id')}")

    # Build legal directed transitions. The orientation is absolute image direction.
    adjacency: dict[tuple[int, int], list[tuple[tuple[int, int], int, int]]] = defaultdict(list)
    for edge in scene.get("edges", []):
        a, b = _rc(edge["a"]), _rc(edge["b"])
        edge_cost = _edge_cost(edge, strategy, mission, weather)
        if edge_cost is None:
            continue
        if edge.get("stairs", False) and not strategy["legged"]:
            continue
        oneway = _rc(edge["oneway_to"]) if edge.get("oneway_to") is not None else None
        for u, v in ((a, b), (b, a)):
            if oneway is not None and oneway != v:
                continue
            adjacency[u].append((v, _step_dir(u, v), edge_cost))

    # State includes heading and whether the required via point has been reached.
    states = [(node, h, passed) for node in nodes for h in range(4) for passed in (0, 1)]
    reverse: dict[tuple[tuple[int, int], int, int], list[tuple[tuple[tuple[int, int], int, int], int]]] = defaultdict(list)
    terminals = set()
    direct_urgent = _is_urgent(mission) and strategy["urgent"] == "direct"
    for node in nodes:
        for old_heading in range(4):
            for passed in (0, 1):
                state = (node, old_heading, passed)
                if passed and node in goal_nodes:
                    terminals.add(state)
                for nxt, direction, step_cost in adjacency.get(node, []):
                    next_passed = int(bool(passed or nxt in via_nodes))
                    turn = 0 if direct_urgent else _turn_cost(strategy, mission, old_heading, direction)
                    next_state = (nxt, direction, next_passed)
                    reverse[next_state].append((state, step_cost + turn))

    # Reverse Dijkstra gives the minimum remaining cost for each state.
    infinity = 10**18
    distance = {state: infinity for state in states}
    heap = []
    for terminal in terminals:
        distance[terminal] = 0
        heapq.heappush(heap, (0, terminal))
    while heap:
        cost, state = heapq.heappop(heap)
        if cost != distance[state]:
            continue
        for previous, weight in reverse.get(state, []):
            candidate = cost + weight
            if candidate < distance[previous]:
                distance[previous] = candidate
                heapq.heappush(heap, (candidate, previous))

    passed_at_start = int(not via_kind or source in via_nodes)
    current = (source, heading, passed_at_start)
    if distance.get(current, infinity) == infinity:
        return None

    # Follow the globally optimal remaining-cost policy; apply tie-break at each junction.
    route: list[int] = []
    seen = set()
    while current not in terminals:
        if current in seen:
            raise RuntimeError(f"Optimal route loop in scene {scene.get('scene_id')}: {current}")
        seen.add(current)
        node, old_heading, passed = current
        options = []
        for nxt, direction, step_cost in adjacency.get(node, []):
            next_passed = int(bool(passed or nxt in via_nodes))
            next_state = (nxt, direction, next_passed)
            remaining = distance.get(next_state, infinity)
            if remaining == infinity:
                continue
            turn = 0 if direct_urgent else _turn_cost(strategy, mission, old_heading, direction)
            options.append((step_cost + turn + remaining, _tie_rank(old_heading, direction), direction, next_state))
        if not options:
            return None
        best_total = min(option[0] for option in options)
        _, _, direction, current = min((o for o in options if o[0] == best_total), key=lambda o: o[1])
        route.append(direction)
        if len(route) > 200:
            raise RuntimeError(f"Route exceeded 200 steps in scene {scene.get('scene_id')}")
    return route


def _behavior_signature(scene: dict[str, Any], strategy: dict[str, Any]) -> tuple[Any, ...]:
    """Collapse policy tuples that induce identical costs on this one mission/map.

    Most family parameters are inactive for a particular demo (for example,
    rainy-cover behavior on a dry map). Caching by effective edge and turn
    costs makes exhaustive fitting much faster without changing results.
    """
    mission = scene["mission"]
    weather = scene.get("weather", "dry")
    edges = tuple(
        (None if edge.get("stairs", False) and not strategy["legged"]
         else _edge_cost(edge, strategy, mission, weather))
        for edge in scene.get("edges", [])
    )
    if _is_urgent(mission) and strategy["urgent"] == "direct":
        turns = (0, 0, 0, 0)
    else:
        turns = tuple(_turn_cost(strategy, mission, h, d)
                      for h, d in ((0, 0), (0, 3), (0, 2), (0, 1)))
    return edges, turns


def _cached_solve(scene: dict[str, Any], strategy: dict[str, Any], cache: dict[tuple[Any, ...], list[int] | None]) -> list[int] | None:
    key = _behavior_signature(scene, strategy)
    if key not in cache:
        cache[key] = solve_route(scene, strategy)
    return cache[key]


def _route_from_demo(demo: dict[str, Any], scene: dict[str, Any]) -> list[int]:
    route = demo.get("route")
    if route is None:
        route = (scene.get("route") or {}).get("moves")
    if route is None:
        raise ValueError(f"Demo {demo.get('scene_id')} has no observed route")
    if isinstance(route, dict):
        route = route.get("moves")
    return list(route)


def filter_strategies(demos: list[dict[str, Any]], scene_by_id: dict[str, dict[str, Any]],
                      candidates: Iterable[dict[str, Any]] | None = None) -> list[dict[str, Any]]:
    """Filter the public policy family to policies matching every demo exactly."""
    survivors = list(all_strategies() if candidates is None else candidates)
    # Route constraints with fewer expected matches first, so later candidates stop early.
    observed = [(demo, scene_by_id[demo["scene_id"]], _route_from_demo(demo, scene_by_id[demo["scene_id"]])) for demo in demos]
    observed.sort(key=lambda item: (len(item[2]), item[0]["scene_id"]))
    for _, scene, true_route in observed:
        cache: dict[tuple[Any, ...], list[int] | None] = {}
        survivors = [strategy for strategy in survivors if _cached_solve(scene, strategy, cache) == true_route]
        if not survivors:
            break
    return survivors


def predict_episode(demos: list[dict[str, Any]], query_scenes: list[dict[str, Any]],
                    scene_by_id: dict[str, dict[str, Any]]) -> tuple[list[list[int]], list[dict[str, Any]]]:
    """Infer the episode policy from demos, then predict all query routes.

    The public task guarantees all surviving strategies agree on every query.
    This function checks that guarantee on the supplied parsed scenes.
    """
    candidates = filter_strategies(demos, scene_by_id)
    if not candidates:
        raise ValueError("No strategy matches the five demo routes; inspect map, mission, or route parsing")
    predictions = []
    for query_scene in query_scenes:
        cache: dict[tuple[Any, ...], list[int] | None] = {}
        options = {tuple(_cached_solve(query_scene, strategy, cache) or []) for strategy in candidates}
        if len(options) != 1:
            raise ValueError(f"Demo-consistent strategies disagree on query {query_scene.get('scene_id')}")
        predictions.append(list(next(iter(options))))
    return predictions, candidates


def validate_dataset(episodes_path: Path, scenes_path: Path, strategies_path: Path) -> dict[str, Any]:
    """Reproduce annotated demo routes using the strategy labels (train/validation QA)."""
    episodes = json.loads(episodes_path.read_text())
    scenes = json.loads(scenes_path.read_text())
    strategies = json.loads(strategies_path.read_text())
    scene_by_id = {scene["scene_id"]: scene for scene in scenes}
    exact = total = 0
    failures = []
    for episode in episodes:
        strategy = strategies[episode["episode_id"]]
        for demo in episode["demos"]:
            scene = scene_by_id[demo["scene_id"]]
            predicted = solve_route(scene, strategy)
            expected = _route_from_demo(demo, scene)
            total += 1
            if predicted == expected:
                exact += 1
            else:
                failures.append({"episode_id": episode["episode_id"], "scene_id": demo["scene_id"],
                                 "predicted": predicted, "expected": expected})
    return {"exact": exact, "total": total, "accuracy": exact / total if total else 0,
            "failures": failures}


def validate_sweep(episodes_path: Path, scenes_path: Path, strategies_path: Path,
                   max_episodes: int | None = None) -> dict[str, Any]:
    """Confirm each labeled policy survives the demo sweep and is query-identifiable.

    This deliberately permits other parameter tuples to survive when they are
    observationally equivalent; the benchmark guarantee is query-route
    identifiability, not necessarily unique parameter identification.
    """
    episodes = json.loads(episodes_path.read_text())
    scenes = json.loads(scenes_path.read_text())
    strategies = json.loads(strategies_path.read_text())
    scene_by_id = {scene["scene_id"]: scene for scene in scenes}
    if max_episodes is not None:
        episodes = episodes[:max_episodes]
    episode_count = 0
    labeled_survives = 0
    query_identifiable = 0
    candidate_counts = []
    failures = []
    for episode in episodes:
        episode_count += 1
        demos = episode["demos"]
        candidates = filter_strategies(demos, scene_by_id)
        candidate_counts.append(len(candidates))
        labeled = strategies[episode["episode_id"]]
        if labeled in candidates:
            labeled_survives += 1
        else:
            failures.append({"episode_id": episode["episode_id"], "error": "labeled strategy was filtered out"})
            continue
        query_ids = [f"{episode['episode_id']}-q{i}" for i in range(6)]
        query_scenes = [scene_by_id[qid] for qid in query_ids if qid in scene_by_id]
        if len(query_scenes) == 6:
            agrees = True
            for query in query_scenes:
                cache: dict[tuple[Any, ...], list[int] | None] = {}
                if len({tuple(_cached_solve(query, s, cache) or []) for s in candidates}) != 1:
                    agrees = False
                    break
            if agrees:
                query_identifiable += 1
            else:
                failures.append({"episode_id": episode["episode_id"], "error": "candidate strategies disagree on a query"})
    return {
        "episodes": episode_count,
        "labeled_strategy_survives": labeled_survives,
        "query_identifiable": query_identifiable,
        "candidate_counts": candidate_counts,
        "failures": failures,
    }


def write_test_predictions(episodes_path: Path, queries_path: Path, parsed_scenes_path: Path,
                           output_path: Path) -> None:
    """Write predictions.json from YOLO/NLU parsed scenes.

    `parsed_scenes.json` must contain a list of scene objects using the public
    scene schema. Demo scene objects need `route.moves` from the route reader;
    query scene objects need graph, robot, weather and grounded mission fields.
    """
    episodes = json.loads(episodes_path.read_text())
    queries = json.loads(queries_path.read_text())
    parsed_scenes = json.loads(parsed_scenes_path.read_text())
    scene_by_id = {scene["scene_id"]: scene for scene in parsed_scenes}
    episode_predictions = {}
    for episode in episodes:
        demos = episode["demos"]
        query_ids = [q["id"] for q in queries if q["episode_id"] == episode["episode_id"]]
        query_scenes = [scene_by_id[qid] for qid in query_ids]
        routes, _ = predict_episode(demos, query_scenes, scene_by_id)
        episode_predictions.update(zip(query_ids, routes))
    ordered = [episode_predictions[q["id"]] for q in queries]
    output_path.write_text(json.dumps(ordered, ensure_ascii=False))


def _zip_json_member(zf: zipfile.ZipFile, split: str, filename: str) -> str:
    """Resolve one delivery_public/{split}/{filename} member without unpacking images."""
    suffix = f"{split}/{filename}"
    names = [name.replace("\\", "/") for name in zf.namelist()]
    matches = [name for name in names if name.endswith(suffix)]
    preferred = [name for name in matches if f"delivery_public/{suffix}" in name]
    matches = preferred or matches
    if not matches:
        raise FileNotFoundError(f"ZIP thiếu {suffix}")
    # Multiple nested copies are ambiguous; fail rather than silently load a wrong split.
    shortest = min(len(name) for name in matches)
    best = [name for name in matches if len(name) == shortest]
    if len(best) != 1:
        raise ValueError(f"Có nhiều member ZIP phù hợp với {suffix}: {best}")
    return best[0]


def _load_zip_jsons(zip_path: Path, split: str, filenames: tuple[str, ...], temp_dir: Path) -> dict[str, Path]:
    """Copy only requested JSON metadata from the archive; never extract its images."""
    temp_dir.mkdir(parents=True, exist_ok=True)
    result = {}
    with zipfile.ZipFile(zip_path, "r") as zf:
        for filename in filenames:
            member = _zip_json_member(zf, split, filename)
            destination = temp_dir / filename
            destination.write_bytes(zf.read(member))
            result[filename] = destination
    return result



In [ ]:
# CELL 11 — Trích ảnh theo yêu cầu từ ZIP và chạy YOLO một lần mỗi scene
from tqdm.auto import tqdm
from ultralytics import YOLO

YOLO_MODEL = YOLO(str(YOLO_MODEL_PATH))
_yolo_names = YOLO_MODEL.names
_yolo_names = [_yolo_names[i] for i in range(len(_yolo_names))] if isinstance(_yolo_names, dict) else list(_yolo_names)
assert _yolo_names == CLASSES, f"YOLO không đúng thứ tự 42 class: {_yolo_names}"
print("YOLO 42 class OK; parameters:", sum(p.numel() for p in YOLO_MODEL.model.parameters()))

with zipfile.ZipFile(DATA_ZIP_PATH, "r") as _zf:
    ZIP_MEMBERS = [n.replace("\\", "/") for n in _zf.namelist()]

def mission_text(value):
    if isinstance(value, str):
        return value.strip()
    if isinstance(value, dict):
        return str(value.get("text") or value.get("mission") or "").strip()
    return ""

def extract_image(split, image_ref, scene_id):
    rel = str(image_ref).replace("\\", "/").lstrip("./")
    if rel.startswith(split + "/"):
        rel = rel[len(split) + 1:]
    suffix = f"/{split}/{rel}"
    matches = [n for n in ZIP_MEMBERS if n.endswith(suffix)]
    if not matches:
        matches = [n for n in ZIP_MEMBERS if n == f"{split}/{rel}"]
    if not matches:
        # Fallback handles image refs that already contain a split/root prefix.
        matches = [n for n in ZIP_MEMBERS if n.endswith("/" + rel) and f"/{split}/" in ("/" + n)]
    preferred = [n for n in matches if f"delivery_public/{split}/" in n]
    matches = preferred or matches
    if len(matches) != 1:
        raise FileNotFoundError(
            f"Không xác định duy nhất ảnh {split}/{image_ref} ({scene_id}); matches={matches[:5]}"
        )
    ext = Path(rel).suffix or ".png"
    out = IMAGE_CACHE_DIR / split / f"{scene_id}{ext}"
    if not out.is_file():
        out.parent.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(DATA_ZIP_PATH, "r") as zf, zf.open(matches[0]) as src, out.open("wb") as dst:
            shutil.copyfileobj(src, dst)
    return out

def detections_for(split, scene_id, image_ref):
    image_path = extract_image(split, image_ref, scene_id)
    dets, _ = predict_dets(YOLO_MODEL, str(image_path), min_conf=min(YOLO_CONF, YOLO_EDGE_CONF, YOLO_ROUTE_CONF))
    return {"image": image_path, "dets": dets}

def select_spatial_rc(graph, place, kind, anchor="none"):
    """Ground NLU spatial references to the detected landmark copy on this graph."""
    candidates = sorted({tuple(int(v) for v in x["rc"]) for x in graph.get("landmarks", []) if x.get("type") == place})
    if not candidates:
        raise ValueError(f"Không phát hiện landmark {place!r} trên {graph.get('scene_id')}")
    if kind in (None, "", "none", "any"):
        return None
    if kind == "north": return min(candidates, key=lambda p: (p[0], p[1]))
    if kind == "south": return max(candidates, key=lambda p: (p[0], -p[1]))
    if kind == "west": return min(candidates, key=lambda p: (p[1], p[0]))
    if kind == "east": return max(candidates, key=lambda p: (p[1], -p[0]))
    if kind in ("near", "far"):
        anchors = sorted({tuple(int(v) for v in x["rc"]) for x in graph.get("landmarks", []) if x.get("type") == anchor})
        if not anchors:
            raise ValueError(f"Không tìm thấy anchor {anchor!r} cho {place!r} trên {graph.get('scene_id')}")
        distances = {p: min(math.hypot(p[0]-a[0], p[1]-a[1]) for a in anchors) for p in candidates}
        if kind == "near": return min(candidates, key=lambda p: (distances[p], p))
        return min(candidates, key=lambda p: (-distances[p], p))
    raise ValueError(f"Loại tham chiếu không hỗ trợ: {kind}")

def graph_from_detections(scene_id, dets, nlu, text, image_path=None):
    """Build the scene graph and retry low-confidence landmark detections if needed."""
    goal = nlu["goal"]
    via = None if nlu["via"] == "none" else nlu["via"]
    goal_kind, goal_anchor = nlu["gkind"], nlu["ganchor"]
    via_kind, via_anchor = nlu["vkind"], nlu["vanchor"]

    needed_places = {goal}
    if via is not None:
        needed_places.add(via)
    if goal_kind in ("near", "far"):
        needed_places.add(goal_anchor)
    if via is not None and via_kind in ("near", "far"):
        needed_places.add(via_anchor)
    needed_places.discard("none")

    def build_base(detection_rows, node_conf):
        graph = dets_to_scene(
            detection_rows, via, goal,
            conf=node_conf,
            edge_conf=YOLO_EDGE_CONF,
            split_long=True,
            route_conf=YOLO_ROUTE_CONF,
        )
        graph["scene_id"] = scene_id
        return graph

    base = build_base(dets, YOLO_CONF)
    found_places = {item["type"] for item in base.get("landmarks", [])}
    missing_places = needed_places - found_places

    if missing_places and image_path is not None:
        # Retry only this image with a lower YOLO confidence. Keep the normal
        # detections and add low-confidence boxes only for required place types.
        retry_dets, _ = predict_dets(
            YOLO_MODEL,
            str(image_path),
            min_conf=0.03,
        )
        retry_places = [
            detection for detection in retry_dets
            if detection["class"] in missing_places
        ]

        if retry_places:
            # Apply the usual thresholds to existing detections first.
            def passes_normal_threshold(detection):
                class_name = detection["class"]
                if is_plain_edge(class_name) or class_name in EDGE_BOX_CLS:
                    threshold = YOLO_EDGE_CONF
                elif class_name.startswith("route_"):
                    threshold = YOLO_ROUTE_CONF
                else:
                    threshold = YOLO_CONF
                return detection["conf"] >= threshold

            normal_dets = [d for d in dets if passes_normal_threshold(d)]
            rescued_keys = {
                (d["class"], tuple(round(v, 1) for v in d["box"]))
                for d in normal_dets
            }
            for detection in retry_places:
                key = (
                    detection["class"],
                    tuple(round(v, 1) for v in detection["box"]),
                )
                if key not in rescued_keys:
                    normal_dets.append(detection)
                    rescued_keys.add(key)

            # Inputs have already been threshold-filtered above; conf=0 lets
            # the parser retain rescued landmarks while edge/route thresholds
            # remain independently enforced.
            base = build_base(normal_dets, 0.0)
            found_places = {item["type"] for item in base.get("landmarks", [])}
            missing_places = needed_places - found_places
            if missing_places:
                base.setdefault("warnings", []).append(
                    "YOLO retry vẫn thiếu landmark cần dùng: "
                    + ", ".join(sorted(missing_places))
                )
            else:
                base.setdefault("warnings", []).append(
                    "Đã khôi phục landmark bằng YOLO retry conf=0.03: "
                    + ", ".join(sorted(needed_places - {
                        item["type"] for item in dets
                    }))
                )

    # Spatial references cannot be grounded correctly if the referenced place
    # or anchor is genuinely absent; raise an explicit error after the retry.
    found_places = {item["type"] for item in base.get("landmarks", [])}
    still_missing = needed_places - found_places
    if still_missing:
        raise ValueError(
            f"{scene_id}: không phát hiện landmark cần thiết "
            f"{sorted(still_missing)} kể cả sau YOLO retry; "
            f"kiểm tra detection/ảnh hoặc NLU output={nlu}"
        )

    goal_rc = select_spatial_rc(base, goal, goal_kind, goal_anchor)
    via_rc = select_spatial_rc(base, via, via_kind, via_anchor) if via is not None else None

    graph = dets_to_scene(
        dets if not missing_places else normal_dets,
        via,
        goal,
        conf=YOLO_CONF if not missing_places else 0.0,
        edge_conf=YOLO_EDGE_CONF,
        split_long=True,
        route_conf=YOLO_ROUTE_CONF,
        mission_goal_rc=list(goal_rc) if goal_rc is not None else None,
        mission_via_rc=list(via_rc) if via_rc is not None else None,
    )
    graph["scene_id"] = scene_id
    graph["mission"] = {
        "text": text,
        "goal": goal,
        "via": via,
        "urgent": nlu["urgent"] == "yes",
        "fragile": nlu["fragile"] == "yes",
        "goal_ref": {
            "kind": goal_kind if goal_rc is not None else "any",
            "anchor": goal_anchor,
            "rc": list(goal_rc) if goal_rc is not None else None,
        },
        "via_ref": {
            "kind": via_kind if via_rc is not None else "any",
            "anchor": via_anchor,
            "rc": list(via_rc) if via_rc is not None else None,
        },
    }
    return graph

print("Image extraction + YOLO + NLU grounding helpers ready.")

In [ ]:
# CELL 12 — FULL VALIDATION: NLU, YOLO graph/route, strategy sweep, end-to-end
from collections import Counter

# A. NLU validation: exact match của 8 head trên 4 safe text views.
VAL_ROWS = expand_views(VAL_BASE)
NLU_VAL_SCORE, NLU_VAL_BY_VIEW, NLU_VAL_PRED = evaluate(VAL_ROWS, VAL_BASE)
print(f"NLU validation exact-8 mean: {NLU_VAL_SCORE:.2%}")

# B. Chạy YOLO đúng một lần cho từng ảnh validation.
VAL_DET_CACHE = {}
for scene in tqdm(VAL_SCENES, desc="YOLO validation images"):
    VAL_DET_CACHE[scene["scene_id"]] = detections_for(
        "validation", scene["scene_id"], scene["image"]
    )

# C. Chấm graph map đầy đủ bằng scene annotations validation.
VAL_GRAPH_PRED = {}
graph_exact = 0
graph_errors = Counter()
for truth in tqdm(VAL_SCENES, desc="compare full graphs"):
    mission = truth.get("mission") or {}
    goal_rc = (mission.get("goal_ref") or {}).get("rc")
    via_rc = (mission.get("via_ref") or {}).get("rc")
    trace = {}
    pred = dets_to_scene(
        VAL_DET_CACHE[truth["scene_id"]]["dets"],
        mission.get("via"), mission.get("goal"),
        conf=YOLO_CONF, edge_conf=YOLO_EDGE_CONF, split_long=True,
        trace=trace, route_conf=YOLO_ROUTE_CONF,
        mission_via_rc=via_rc, mission_goal_rc=goal_rc,
    )
    pred["scene_id"] = truth["scene_id"]
    VAL_GRAPH_PRED[truth["scene_id"]] = pred
    errors, _ = compare_scenes(pred, truth, trace.get("edge_src", {}))
    graph_exact += int(not errors)
    graph_errors.update(x["type"] for x in errors)
print(f"YOLO full-graph exact: {graph_exact}/{len(VAL_SCENES)} = {graph_exact/max(1,len(VAL_SCENES)):.2%}")
print("Graph error counts:", dict(graph_errors))

# D. Chấm route reader trên toàn bộ demo validation.
val_demo_scenes = [s for s in VAL_SCENES if (s.get("route") or {}).get("moves") is not None]
route_exact = 0
route_step_ok = route_step_n = route_tp = route_fp = route_fn = 0
route_failures = []
for truth in tqdm(val_demo_scenes, desc="YOLO demo routes"):
    pred = VAL_GRAPH_PRED[truth["scene_id"]]
    true_moves = list((truth.get("route") or {}).get("moves") or [])
    got = list(pred.get("route_moves") or [])
    route_exact += int(got == true_moves)
    common = min(len(got), len(true_moves))
    route_step_ok += sum(got[i] == true_moves[i] for i in range(common))
    route_step_n += max(len(got), len(true_moves))
    true_nodes = (truth.get("route") or {}).get("nodes") or []
    true_edges = {(tuple(a), tuple(b)) for a,b in zip(true_nodes, true_nodes[1:])}
    pred_edges = {(tuple(e["from"]),tuple(e["to"])) for e in pred.get("route_edges",[])}
    route_tp += len(true_edges & pred_edges); route_fp += len(pred_edges-true_edges); route_fn += len(true_edges-pred_edges)
    if got != true_moves:
        route_failures.append({"scene_id":truth["scene_id"],"true":true_moves,"pred":got,
                               "reconstruction":pred.get("route_reconstruction"),"warnings":pred.get("warnings",[])})
route_p=route_tp/max(1,route_tp+route_fp); route_r=route_tp/max(1,route_tp+route_fn)
route_f1=2*route_p*route_r/max(1e-12,route_p+route_r)
print(f"YOLO route exact: {route_exact}/{len(val_demo_scenes)} = {route_exact/max(1,len(val_demo_scenes)):.2%}")
print(f"Route step score={route_step_ok/max(1,route_step_n):.2%}; directed edge P/R/F1={route_p:.4f}/{route_r:.4f}/{route_f1:.4f}")
print("Route failure examples:", route_failures[:10])

# E. Oracle simulator audit and blind strategy sweep on annotated validation scenes.
# strategies.json and labels are used only for validation/audit, never for fitting test episodes.
def audit_demo_routes(episodes, scenes, strategies):
    """Audit the public simulator against GT demo routes and labeled policies."""
    scene_by_id = {scene["scene_id"]: scene for scene in scenes}
    exact = total = 0
    failures = []

    for episode in episodes:
        episode_id = episode["episode_id"]
        strategy = strategies.get(episode_id) if isinstance(strategies, dict) else None
        if not isinstance(strategy, dict):
            failures.append({"episode_id": episode_id, "error": "missing strategy entry"})
            continue

        for demo in episode.get("demos", []):
            scene_id = demo.get("scene_id")
            scene = scene_by_id.get(scene_id)
            if scene is None:
                failures.append({"episode_id": episode_id, "scene_id": scene_id,
                                 "error": "demo scene missing from scenes"})
                continue
            try:
                expected = _route_from_demo(demo, scene)
                predicted = solve_route(scene, strategy)
            except Exception as exc:
                total += 1
                failures.append({"episode_id": episode_id, "scene_id": scene_id,
                                 "error": f"simulator exception: {type(exc).__name__}: {exc}"})
                continue
            total += 1
            if predicted == expected:
                exact += 1
            else:
                failures.append({"episode_id": episode_id, "scene_id": scene_id,
                                 "predicted": predicted, "expected": expected})

    return {"exact": exact, "total": total,
            "accuracy": exact / total if total else 0.0, "failures": failures}

# E. Oracle simulator audit and blind strategy sweep on annotated validation scenes.
# strategies.json is used only for audit (not to fit candidates); labels are used only to score.
DEMO_AUDIT = audit_demo_routes(VAL_EPISODES, VAL_SCENES, VAL_STRATEGIES)
print(f"Oracle strategy demo audit: {DEMO_AUDIT['exact']}/{DEMO_AUDIT['total']} = {DEMO_AUDIT['accuracy']:.2%}")
VAL_SCENE_BY_ID = {s["scene_id"]: s for s in VAL_SCENES}
label_by_query = {q["id"]: list(VAL_LABELS[i]) for i,q in enumerate(VAL_QUERIES)}
blind_exact = blind_total = 0
blind_failures = []
candidate_sizes = []
for episode in tqdm(VAL_EPISODES, desc="strategy sweep on GT graphs"):
    candidates = filter_strategies(episode["demos"], VAL_SCENE_BY_ID)
    candidate_sizes.append(len(candidates))
    if VAL_STRATEGIES.get(episode["episode_id"]) not in candidates:
        blind_failures.append({"episode_id":episode["episode_id"],"error":"labeled strategy filtered out"})
    qs = [q for q in VAL_QUERIES if q["episode_id"] == episode["episode_id"]]
    for q in qs:
        scene = VAL_SCENE_BY_ID[q["id"]]
        cache={}
        options={tuple(_cached_solve(scene,st,cache) or []) for st in candidates}
        pred=list(next(iter(options))) if len(options)==1 else None
        blind_total += 1
        blind_exact += int(pred == label_by_query[q["id"]])
        if pred != label_by_query[q["id"]]:
            blind_failures.append({"episode_id":episode["episode_id"],"query_id":q["id"],
                                  "candidate_routes":len(options),"pred":pred,"true":label_by_query[q["id"]]})
print(f"Blind sweep on GT graphs: {blind_exact}/{blind_total} = {blind_exact/max(1,blind_total):.2%}")
print(f"Candidates/episode: mean={statistics.mean(candidate_sizes) if candidate_sizes else 0:.1f}, min={min(candidate_sizes,default=0)}, max={max(candidate_sizes,default=0)}")
print("Blind sweep failures:", blind_failures[:10])

# F. Full end-to-end validation: YOLO image + NLU mission + demo route fit + query solve.
val_texts = [mission_text((s.get("mission") or {}).get("text")) for s in VAL_SCENES]
val_nlu_predictions = predict_rows([{"text":t} for t in val_texts])
VAL_E2E_SCENES = {}
for idx, (truth, parsed_nlu) in enumerate(tqdm(zip(VAL_SCENES,val_nlu_predictions), total=len(VAL_SCENES), desc="build e2e val scenes")):
    sid=truth["scene_id"]
    VAL_E2E_SCENES[sid] = graph_from_detections(sid, VAL_DET_CACHE[sid]["dets"], parsed_nlu, val_texts[idx])

# Fallback ranking is used only if visual demo routes leave no exact strategy survivors.
def best_effort_strategies(demos, scene_by_id):
    usable=[]
    for demo in demos:
        route=demo.get("route")
        if route is not None and len(route)>0 and demo["scene_id"] in scene_by_id:
            usable.append((demo,scene_by_id[demo["scene_id"]],list(route)))
    if not usable:
        return [next(iter(all_strategies()))], 0
    score_cache=[{} for _ in usable]
    best_score=-1; best=[]
    for strategy in all_strategies():
        score=0
        for i,(_,scene,observed) in enumerate(usable):
            score += int(_cached_solve(scene,strategy,score_cache[i]) == observed)
        if score > best_score:
            best_score=score; best=[strategy]
        elif score == best_score:
            best.append(strategy)
    return best, best_score

def predict_queries_from_demos(demos, query_scenes, scene_by_id):
    candidates=filter_strategies(demos,scene_by_id)
    fit_mode="exact"
    if not candidates:
        candidates,score=best_effort_strategies(demos,scene_by_id)
        fit_mode=f"best_effort_{score}_of_{len(demos)}"
    preds=[]; uncertain=0
    for query_scene in query_scenes:
        cache={}
        options={tuple(_cached_solve(query_scene,st,cache) or []) for st in candidates}
        if len(options)>1: uncertain += 1
        preds.append(list(sorted(options,key=lambda x:(len(x),x))[0]) if options else [])
    return preds,candidates,fit_mode,uncertain

VAL_E2E_PRED_BY_ID={}; e2e_failures=[]; e2e_candidate_counts=[]; e2e_uncertain=0
for episode in tqdm(VAL_EPISODES,desc="end-to-end strategy fit"):
    demos=[]
    for demo in episode["demos"]:
        parsed=VAL_E2E_SCENES.get(demo["scene_id"])
        demos.append({**demo,"route":list((parsed or {}).get("route_moves") or [])})
    qrows=[q for q in VAL_QUERIES if q["episode_id"]==episode["episode_id"]]
    qscenes=[VAL_E2E_SCENES[q["id"]] for q in qrows]
    pred_routes,cands,mode,uncertain=predict_queries_from_demos(demos,qscenes,VAL_E2E_SCENES)
    e2e_candidate_counts.append(len(cands)); e2e_uncertain += uncertain
    for q,route in zip(qrows,pred_routes):
        VAL_E2E_PRED_BY_ID[q["id"]]=route
        if route != label_by_query[q["id"]]:
            e2e_failures.append({"episode_id":episode["episode_id"],"query_id":q["id"],"fit_mode":mode,
                                 "route":route,"true":label_by_query[q["id"]],"candidates":len(cands)})
E2E_EXACT=sum(VAL_E2E_PRED_BY_ID.get(q["id"])==label_by_query[q["id"]] for q in VAL_QUERIES)
print(f"END-TO-END validation exact: {E2E_EXACT}/{len(VAL_QUERIES)} = {E2E_EXACT/max(1,len(VAL_QUERIES)):.2%}")
print(f"E2E unresolved candidate disagreements={e2e_uncertain}; failures={len(e2e_failures)}")
print("E2E failure examples:",e2e_failures[:15])

VALIDATION_REPORT={
    "nlu_exact8_mean":NLU_VAL_SCORE,"nlu_by_view":NLU_VAL_BY_VIEW,
    "yolo_graph_exact":graph_exact,"yolo_graph_total":len(VAL_SCENES),"yolo_graph_errors":dict(graph_errors),
    "yolo_route_exact":route_exact,"yolo_route_total":len(val_demo_scenes),"yolo_route_step_score":route_step_ok/max(1,route_step_n),
    "yolo_route_edge_f1":route_f1,"oracle_demo_audit":DEMO_AUDIT,
    "blind_sweep_exact":blind_exact,"blind_sweep_total":blind_total,
    "end_to_end_exact":E2E_EXACT,"end_to_end_total":len(VAL_QUERIES),
    "end_to_end_failures":e2e_failures,
}
(OUTPUT_DIR/"validation_report.json").write_text(json.dumps(VALIDATION_REPORT,ensure_ascii=False,indent=2),encoding="utf-8")
(OUTPUT_DIR/"validation_e2e_predictions.json").write_text(
    json.dumps([VAL_E2E_PRED_BY_ID.get(q["id"],[]) for q in VAL_QUERIES],ensure_ascii=False),encoding="utf-8")
print("Saved validation report and predictions in",OUTPUT_DIR.resolve())

In [ ]:
# CELL 13 — Test inference tối ưu: nạp ZIP một lần, YOLO theo batch
# Nếu CELL 13 cũ đang chạy, dừng cell đó rồi chạy lại cell này.
from collections import defaultdict
from pathlib import Path
from tqdm.auto import tqdm
import zipfile
import shutil
import json
import torch

# Test không có scenes/labels/strategies; chỉ dùng episodes, queries, ảnh,
# mission text và route demo YOLO đọc từ ảnh.
def collect_requests(episodes, queries):
    requests = []

    for episode in episodes:
        for i, demo in enumerate(episode["demos"]):
            scene_id = demo.get("scene_id") or f"{episode['episode_id']}-d{i}"
            requests.append({
                "scene_id": scene_id,
                "episode_id": episode["episode_id"],
                "image": demo["image"],
                "text": mission_text(demo.get("mission")),
                "kind": "demo",
            })

    for query in queries:
        requests.append({
            "scene_id": query["id"],
            "episode_id": query["episode_id"],
            "image": query["image"],
            "text": mission_text(query.get("mission")),
            "kind": "query",
        })

    for row in requests:
        if not row["text"]:
            raise ValueError(f"Mission rỗng ở {row['scene_id']}")

    return requests


TEST_REQUESTS = collect_requests(TEST_EPISODES, TEST_QUERIES)
print(
    "Scenes to parse:", len(TEST_REQUESTS),
    "| demos:", sum(row["kind"] == "demo" for row in TEST_REQUESTS),
    "| queries:", len(TEST_QUERIES),
)

# NLU chạy batch; không gọi lại model NLP từng scene.
test_nlu_outputs = predict_rows([
    {"text": row["text"]}
    for row in TEST_REQUESTS
])
NLU_BY_SCENE = {
    row["scene_id"]: nlu
    for row, nlu in zip(TEST_REQUESTS, test_nlu_outputs)
}

# Tạo chỉ mục ZIP một lần thay vì quét ZIP_MEMBERS và mở ZIP cho từng ảnh.
# Key ví dụ: /test/images/abc.png -> đường dẫn đầy đủ bên trong ZIP.
TEST_ZIP_INDEX = defaultdict(list)
for member in ZIP_MEMBERS:
    normalized = member.replace("\\", "/").lstrip("/")
    parts = normalized.split("/")
    for i, part in enumerate(parts):
        if part == "test":
            TEST_ZIP_INDEX["/" + "/".join(parts[i:])].append(member)


def resolve_test_image_member(image_ref, scene_id):
    rel = str(image_ref).replace("\\", "/").lstrip("./")
    for prefix in ("delivery_public/test/", "test/"):
        if rel.startswith(prefix):
            rel = rel[len(prefix):]

    matches = TEST_ZIP_INDEX.get("/test/" + rel, [])
    if not matches:
        # Fallback cho metadata chứa đường dẫn có thư mục bọc bổ sung.
        matches = [
            member for member in ZIP_MEMBERS
            if member.replace("\\", "/").endswith("/" + rel)
            and "/test/" in ("/" + member.replace("\\", "/"))
        ]

    preferred = [member for member in matches if "delivery_public/test/" in member]
    matches = preferred or matches

    if len(matches) != 1:
        raise FileNotFoundError(
            f"Không xác định duy nhất ảnh test {image_ref!r} ({scene_id}); "
            f"matches={matches[:5]}"
        )
    return matches[0]


# Chuẩn bị toàn bộ ảnh bằng một ZipFile handle, giảm overhead mở ZIP lặp lại.
TEST_IMAGE_BY_ID = {}
with zipfile.ZipFile(DATA_ZIP_PATH, "r") as test_zip:
    for row in tqdm(TEST_REQUESTS, desc="extract test images"):
        scene_id = row["scene_id"]
        rel = str(row["image"]).replace("\\", "/").lstrip("./")
        suffix = Path(rel).suffix or ".png"
        image_path = IMAGE_CACHE_DIR / "test" / f"{scene_id}{suffix}"
        image_path.parent.mkdir(parents=True, exist_ok=True)

        if not image_path.is_file():
            member = resolve_test_image_member(row["image"], scene_id)
            with test_zip.open(member, "r") as src, image_path.open("wb") as dst:
                shutil.copyfileobj(src, dst, length=1024 * 1024)

        TEST_IMAGE_BY_ID[scene_id] = image_path

# YOLO batch inference. Giữ nguyên imgsz/conf và không bật augmentation để
# kết quả gần với validation trước; batch chỉ giảm overhead từng lần predict.
TEST_BATCH_SIZE = 16 if torch.cuda.is_available() else 2
TEST_DEVICE = 0 if torch.cuda.is_available() else "cpu"
TEST_PATH_TO_ID = {
    str(path.resolve()): scene_id
    for scene_id, path in TEST_IMAGE_BY_ID.items()
}

TEST_SCENE_BY_ID = {}
TEST_DET_BY_ID = {}
source_paths = [str(TEST_IMAGE_BY_ID[row["scene_id"]]) for row in TEST_REQUESTS]
result_stream = YOLO_MODEL.predict(
    source=source_paths,
    stream=True,
    batch=TEST_BATCH_SIZE,
    imgsz=YOLO_IMGSZ,
    conf=min(YOLO_CONF, YOLO_EDGE_CONF, YOLO_ROUTE_CONF),
    device=TEST_DEVICE,
    half=torch.cuda.is_available(),
    workers=4 if torch.cuda.is_available() else 0,
    verbose=False,
)

for result in tqdm(result_stream, total=len(TEST_REQUESTS), desc="YOLO batch + graph/NLU"):
    result_path = str(Path(result.path).resolve())
    scene_id = TEST_PATH_TO_ID.get(result_path)
    if scene_id is None:
        raise KeyError(f"YOLO trả về ảnh không nằm trong input: {result.path}")

    detections = []
    boxes = result.boxes
    if boxes is not None:
        xyxy = boxes.xyxy.cpu().numpy()
        class_ids = boxes.cls.cpu().numpy()
        confidences = boxes.conf.cpu().numpy()

        for box, class_id, confidence in zip(xyxy, class_ids, confidences):
            x1, y1, x2, y2 = map(float, box)
            detections.append({
                "class": CLASSES[int(class_id)],
                "box": [x1, y1, x2, y2],
                "conf": float(confidence),
                "center": ((x1 + x2) / 2, (y1 + y2) / 2),
                "w": x2 - x1,
                "h": y2 - y1,
            })

    TEST_DET_BY_ID[scene_id] = detections

if len(TEST_DET_BY_ID) != len(TEST_REQUESTS):
    missing = [
        row["scene_id"] for row in TEST_REQUESTS
        if row["scene_id"] not in TEST_DET_BY_ID
    ]
    raise RuntimeError(f"YOLO thiếu kết quả cho {len(missing)} scene: {missing[:10]}")

# Parse graph after the YOLO stream is complete. This also allows a safe,
# isolated low-confidence YOLO retry for scenes missing a required landmark.
request_by_id = {row["scene_id"]: row for row in TEST_REQUESTS}
for scene_id in tqdm(TEST_DET_BY_ID, desc="ground missions + reconstruct routes"):
    request = request_by_id[scene_id]
    parsed = graph_from_detections(
        scene_id,
        TEST_DET_BY_ID[scene_id],
        NLU_BY_SCENE[scene_id],
        request["text"],
        image_path=TEST_IMAGE_BY_ID[scene_id],
    )
    TEST_SCENE_BY_ID[scene_id] = parsed

# Fit policy riêng từng episode từ 5 route demo rồi dự đoán các query.
TEST_PRED_BY_ID = {}
TEST_FIT_REPORT = []

for episode in tqdm(TEST_EPISODES, desc="fit hidden strategies from 5 demos"):
    demos = []
    for i, demo in enumerate(episode["demos"]):
        scene_id = demo.get("scene_id") or f"{episode['episode_id']}-d{i}"
        parsed = TEST_SCENE_BY_ID[scene_id]
        demos.append({
            **demo,
            "scene_id": scene_id,
            "route": list(parsed.get("route_moves") or []),
        })

    episode_queries = [
        query for query in TEST_QUERIES
        if query["episode_id"] == episode["episode_id"]
    ]
    query_scenes = [
        TEST_SCENE_BY_ID[query["id"]]
        for query in episode_queries
    ]

    routes, candidates, fit_mode, uncertain = predict_queries_from_demos(
        demos,
        query_scenes,
        TEST_SCENE_BY_ID,
    )

    for query, route in zip(episode_queries, routes):
        TEST_PRED_BY_ID[query["id"]] = route

    TEST_FIT_REPORT.append({
        "episode_id": episode["episode_id"],
        "fit_mode": fit_mode,
        "candidate_count": len(candidates),
        "query_disagreements": uncertain,
        "demo_route_lengths": [len(demo.get("route") or []) for demo in demos],
        "demo_route_reconstruction": [
            TEST_SCENE_BY_ID[demo["scene_id"]].get("route_reconstruction")
            for demo in demos
        ],
    })

# Đảm bảo đúng thứ tự queries.json và đúng số phần tử.
PREDICTIONS = [
    TEST_PRED_BY_ID.get(query["id"], [])
    for query in TEST_QUERIES
]
assert len(PREDICTIONS) == len(TEST_QUERIES)

invalid = [
    {"query_id": query["id"], "length": len(route)}
    for query, route in zip(TEST_QUERIES, PREDICTIONS)
    if not (3 <= len(route) <= 45)
]

PREDICTIONS_PATH = OUTPUT_DIR / "predictions.json"
PREDICTIONS_PATH.write_text(
    json.dumps(PREDICTIONS, ensure_ascii=False),
    encoding="utf-8",
)
(OUTPUT_DIR / "test_fit_report.json").write_text(
    json.dumps(TEST_FIT_REPORT, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
(OUTPUT_DIR / "test_parsed_scenes.json").write_text(
    json.dumps(list(TEST_SCENE_BY_ID.values()), ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("\nTest inference complete")
print("queries:", len(TEST_QUERIES), "| prediction rows:", len(PREDICTIONS))
print("invalid route lengths (expected 3..45):", len(invalid))
if invalid:
    print(invalid[:20])
print(
    "episodes without exact demo fit:",
    sum(row["fit_mode"] != "exact" for row in TEST_FIT_REPORT),
)
print(
    "episodes with query disagreement:",
    sum(row["query_disagreements"] > 0 for row in TEST_FIT_REPORT),
)
print("predictions.json:", PREDICTIONS_PATH.resolve())
print("Sample predictions:", PREDICTIONS[:5])
